# Figure 5 — annotation-node-filtered H → W → H–W submission notebook

The continuous 50-factor H matrix is shown first, W supplies biological interpretation,
and the aligned factors connect K=8 programs to the five retained epithelial cell types.
All categorical analysis uses `scknife_final_celltype` and exponential-decay node selection.

## Plot-only execution contract

This notebook renders existing intermediate numerical results only. It does not recompute upstream analysis.


In [ ]:
PLOT_ONLY_FIGURE = "Figure_5"
ANALYSIS_RECOMPUTED = False
ALLOWED_INPUT_SUFFIXES = {".h5ad", ".h5", ".csv", ".gz", ".parquet", ".json"}
print(PLOT_ONLY_FIGURE, "plot-only mode; analysis_recomputed=", ANALYSIS_RECOMPUTED)


In [ ]:
# Conventional Jupyter inline plotting: live Matplotlib figures are rendered by plt.show().
get_ipython().run_line_magic('matplotlib', 'inline')

# Epithelial scKNIFE analysis: H → W → cell-type connections

1. **H first:** summarize the 50 continuous latent factors across the five retained epithelial cell types.
2. **W next:** group the aligned factors into K=8 programs and interpret them with exponential-decay-selected nodes.
3. **H–W connections:** reconstruct program and selected-node activity without introducing any additional cell grouping.

## 0. Imports, constants, and reproducibility settings

The shared latent-factor axis is always 50-dimensional. `W` and `H` use the same factor order, which is what later permits W×H reconstruction.

In [ ]:
import json
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from matplotlib.colors import ListedColormap
from scipy.cluster.hierarchy import fcluster, leaves_list, linkage
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import squareform
from scipy.stats import kruskal, mannwhitneyu
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

NOTEBOOK_DIR = Path.cwd().resolve()
INPUT_DIR = NOTEBOOK_DIR / "data/Figure_5"
W_PATH = INPUT_DIR / "inhouse_epithelial_W_graph.parquet"
NODE_METADATA_PATH = INPUT_DIR / "node_metadata.csv"
CELL_METADATA_PATH = INPUT_DIR / "epithelial_H_cell_metadata.parquet"

SEED = 42
BOOTSTRAPS = 200
RESAMPLE_FRACTION = 0.80
TOP_N = 20
DECAY_TAU = 5.0
DECAY_CUMULATIVE_MASS = 0.95
MAX_NODES_PER_CLUSTER_KIND = 20
K_VALUES = tuple(range(5, 21))
EXPECTED_W_SHAPE = (39247, 50)
EXPECTED_H_SHAPE = (43574, 50)
EXCLUDED_W_NODE_KINDS = ("cell_type", "cell_state")
EXPECTED_EXCLUDED_NODE_KIND_COUNTS = {"cell_state": 83, "cell_type": 95}
EXPECTED_ELIGIBLE_W_NODES = 39035

R5_EXPORT_DPI = 300
R5_FONT_FAMILY = "DejaVu Sans"
R5_PDF_FONT_TYPE = 42
R5_SVG_FONT_TYPE = "none"
R5_VECTOR_TEXT_RCPARAMS = {
    "font.family": R5_FONT_FAMILY,
    "pdf.fonttype": R5_PDF_FONT_TYPE,
    "svg.fonttype": R5_SVG_FONT_TYPE,
}

CLUSTER_COLORS = [
    "#4477AA", "#EE7733", "#228833", "#CC3311", "#AA3377",
    "#66CCEE", "#CCBB44", "#EE6677", "#BBBBBB", "#000000",
]
CLUSTER_COLORS = [
    matplotlib.colors.to_hex(plt.get_cmap("tab20")(value))
    for value in np.linspace(0.0, 0.95, 20)
]
CATEGORY_COLORS = {
    name: matplotlib.colors.to_hex(plt.get_cmap("tab10")(i))
    for i, name in enumerate([
        "gene", "reaction", "GO_term", "metabolite", "cellchat_entity",
        "task", "pathway", "cell_state", "geneset",
    ])
}

plt.rcParams.update({
    "font.family": R5_FONT_FAMILY, "font.size": 8, "axes.grid": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 120,
})

ARTIFACTS = {}

def now():
    return datetime.now(timezone.utc).strftime("%H:%M:%S UTC")

def progress(message):
    print(f"[{now()}] {message}", flush=True)

def show_table(table, title, rows=20):
    display(HTML(f"<h4>{title}</h4>"))
    display(table.head(rows))
    if len(table) > rows:
        print(f"Showing {rows:,} of {len(table):,} rows.", flush=True)

progress(f"Kernel ready: Python {platform.python_version()}")

## 1. Load and validate W, the numerical H matrix, and fixed epithelial cell types

In [ ]:
progress("Loading canonical W, numerical H activity, and fixed epithelial cell types")
required = [W_PATH, NODE_METADATA_PATH, CELL_METADATA_PATH]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing required inputs: {missing}")

W = pd.read_parquet(W_PATH)
W.columns = W.columns.astype(str)
node_metadata = pd.read_csv(NODE_METADATA_PATH)
cell_metadata = pd.read_parquet(CELL_METADATA_PATH)
H_COLUMNS = [f"H_graph_{i}" for i in range(50)]
H = cell_metadata[H_COLUMNS].to_numpy(dtype=np.float64)
CELLTYPE_COLUMN = "scknife_final_celltype"
CELLTYPE_ORDER = [
    "Epithelial progenitor", "Epithelial luminal", "Epithelial basal",
    "Epithelial cycling", "IFN-response epithelial",
]
CELLTYPE_LABELS = cell_metadata[CELLTYPE_COLUMN].astype(str)

preflight = pd.Series({
    "W_shape_39247x50": W.shape == EXPECTED_W_SHAPE,
    "W_unique_nodes": W.index.is_unique,
    "W_finite": np.isfinite(W.to_numpy()).all(),
    "node_metadata_aligned": node_metadata["node_name"].astype(str).tolist() == W.index.astype(str).tolist(),
    "H_shape_43574x50": H.shape == EXPECTED_H_SHAPE,
    "H_unique_cells": cell_metadata["cell_join_key"].is_unique,
    "H_finite": np.isfinite(H).all(),
    "five_fixed_epithelial_celltypes": set(CELLTYPE_LABELS) == set(CELLTYPE_ORDER),
}, name="passed")
display(preflight.to_frame())
if not preflight.all():
    raise RuntimeError("Input preflight failed")

NONZERO = W.to_numpy().sum(axis=1) > 0
NODE_KIND_ALLOWED = ~node_metadata["node_kind"].astype(str).isin(EXCLUDED_W_NODE_KINDS)
ELIGIBLE_MASK = NONZERO & NODE_KIND_ALLOWED.to_numpy()
EXCLUDED_NODE_KIND_COUNTS = node_metadata.loc[~NODE_KIND_ALLOWED, "node_kind"].astype(str).value_counts().sort_index()
W_ELIGIBLE = W.loc[ELIGIBLE_MASK]
NODE_METADATA_ELIGIBLE = node_metadata.loc[ELIGIBLE_MASK].reset_index(drop=True)
progress(f"Preflight passed: {W.shape[0]:,} W nodes, {H.shape[0]:,} cells, {H.shape[1]} matched factors")
progress(f"Consensus input: {ELIGIBLE_MASK.sum():,} eligible nodes; {int((~NODE_KIND_ALLOWED).sum()):,} cell-type/cell-state annotation nodes and {int((~NONZERO).sum()):,} zero-signal rows excluded")
ARTIFACTS["preflight"] = preflight
ARTIFACTS["excluded_node_kind_counts"] = EXCLUDED_NODE_KIND_COUNTS

# Part I — H: where are the continuous factors active?

H contains continuous usage values for 50 aligned factors across 43,574 epithelial cells.
The factors are first summarized using only the five retained `scknife_final_celltype` identities;
W later supplies their biological interpretation and shared factor-group ordering.

## 2. Fixed epithelial cell-type census

In [ ]:
CELLTYPE_COUNTS = CELLTYPE_LABELS.value_counts().reindex(CELLTYPE_ORDER, fill_value=0).rename("cells")
CELLTYPE_PERCENT = CELLTYPE_COUNTS.div(CELLTYPE_COUNTS.sum()).mul(100).rename("percent")
CELLTYPE_CENSUS = pd.concat([CELLTYPE_COUNTS, CELLTYPE_PERCENT], axis=1)
display(CELLTYPE_CENSUS)
ARTIFACTS["celltype_census"] = CELLTYPE_CENSUS

## 3. H-factor summaries by fixed epithelial cell type

The 50 continuous H-factor activities are summarized using the retained cell-type identities.

In [ ]:
def row_log_z(table):
    values = np.log1p(np.asarray(table, dtype=float))
    return (values - values.mean(axis=1, keepdims=True)) / np.maximum(
        values.std(axis=1, keepdims=True), 1e-12
    )


def inline_heatmap(table, title, color_label):
    values = np.asarray(table, dtype=float)
    limit = max(float(np.nanpercentile(np.abs(values), 98)), 1e-6)
    fig, axis = plt.subplots(figsize=(7.5, max(4, 0.22 * len(table))), constrained_layout=True)
    image = axis.imshow(values, aspect="auto", cmap="RdBu_r", vmin=-limit, vmax=limit,
                        interpolation="nearest", rasterized=True)
    axis.set_title(title, loc="left", fontweight="bold")
    axis.set_xticks(range(table.shape[1]))
    axis.set_xticklabels(table.columns, rotation=45, ha="right", fontsize=7)
    axis.set_yticks(range(table.shape[0]))
    axis.set_yticklabels(table.index, fontsize=6)
    fig.colorbar(image, ax=axis, label=color_label)
    display(fig); plt.close(fig)

H_CELLTYPE_MEAN = pd.DataFrame(H, columns=H_COLUMNS).groupby(
    CELLTYPE_LABELS.reset_index(drop=True), observed=True
).mean().reindex(CELLTYPE_ORDER).T
H_CELLTYPE_LOG_Z = pd.DataFrame(
    row_log_z(H_CELLTYPE_MEAN), index=H_COLUMNS, columns=CELLTYPE_ORDER
)
inline_heatmap(H_CELLTYPE_LOG_Z, "Mean H-factor activity by epithelial cell type",
               "Row z-score of log(1 + mean H)")
ARTIFACTS["H_celltype_mean"] = H_CELLTYPE_MEAN
ARTIFACTS["H_celltype_log_z"] = H_CELLTYPE_LOG_Z

# Part II — W: what do the latent factors represent?

W describes the biological content of each latent factor. Rows are biological nodes and columns are the same 50 factors used by H.

The next sections cluster factors with similar W profiles, then identify the biological nodes that best characterize each component cluster.

## 4. Consensus-clustering helpers

Each nonzero W node is transformed with `log1p` and z-scored across the 50 factors. Consensus clustering repeatedly resamples W nodes and measures how consistently pairs of factors cluster together.

In [ ]:
def nodewise_zscore(values):
    logged = np.log1p(np.asarray(values, dtype=np.float64))
    return (logged - logged.mean(axis=1, keepdims=True)) / np.maximum(
        logged.std(axis=1, keepdims=True), 1e-12
    )

def consensus_sweep(node_zscores, component_names):
    rng = np.random.default_rng(SEED)
    sums = {k: np.zeros((len(component_names), len(component_names)), dtype=float) for k in K_VALUES}
    sample_size = int(RESAMPLE_FRACTION * len(node_zscores))
    started = time.time()
    progress(f"Starting k=5–20 consensus sweep: {BOOTSTRAPS} bootstraps × {sample_size:,} sampled nodes")
    for bootstrap in range(BOOTSTRAPS):
        rows = rng.choice(len(node_zscores), size=sample_size, replace=True)
        correlation = np.corrcoef(node_zscores[rows].T)
        distance = np.clip(1 - correlation, 0, 2)
        np.fill_diagonal(distance, 0)
        for k in K_VALUES:
            labels = AgglomerativeClustering(
                n_clusters=k, metric="precomputed", linkage="average"
            ).fit_predict(distance)
            sums[k] += labels[:, None] == labels[None, :]
        if (bootstrap + 1) % 20 == 0 or bootstrap == 0:
            progress(f"Consensus bootstrap {bootstrap + 1}/{BOOTSTRAPS}")

    records, solutions = [], {}
    for k in K_VALUES:
        consensus = sums[k] / BOOTSTRAPS
        distance = 1 - consensus
        np.fill_diagonal(distance, 0)
        tree = linkage(squareform(distance, checks=False), method="average")
        labels = fcluster(tree, t=k, criterion="maxclust") - 1
        order = leaves_list(tree)
        relabel = {old: new for new, old in enumerate(pd.unique(labels[order]))}
        labels = np.asarray([relabel[label] for label in labels], dtype=int)
        upper = consensus[np.triu_indices(len(consensus), 1)]
        sizes = np.bincount(labels)
        records.append({
            "k": k,
            "PAC_0.1_0.9": float(np.mean((upper > 0.1) & (upper < 0.9))),
            "consensus_silhouette": float(silhouette_score(distance, labels, metric="precomputed")),
            "minimum_cluster_size": int(sizes.min()),
            "maximum_cluster_size": int(sizes.max()),
            "no_singletons": bool(sizes.min() >= 2),
        })
        solutions[k] = {
            "consensus": consensus, "distance": distance, "tree": tree,
            "labels": labels, "leaf_order": order,
        }
    progress(f"Consensus sweep complete in {(time.time() - started) / 60:.1f} minutes")
    return pd.DataFrame(records), solutions

def plot_consensus_diagnostics(metrics):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
    axes[0].plot(metrics["k"], metrics["PAC_0.1_0.9"], marker="o", color="#4477AA")
    axes[0].set(title="A  Consensus ambiguity", xlabel="k", ylabel="PAC (lower is better)")
    axes[1].plot(metrics["k"], metrics["consensus_silhouette"], marker="o", color="#CC3311")
    axes[1].set(title="B  Consensus separation", xlabel="k", ylabel="Silhouette (higher is better)")
    for axis in axes:
        axis.set_xticks(K_VALUES)
    valid = metrics[metrics["no_singletons"]]
    best = valid.sort_values(
        ["consensus_silhouette", "PAC_0.1_0.9"], ascending=[False, True]
    ).iloc[0]
    for axis, column in zip(axes, ["PAC_0.1_0.9", "consensus_silhouette"]):
        axis.scatter([best["k"]], [best[column]], s=90, facecolors="none",
                     edgecolors="#000000", linewidths=1.5, label="Best non-singleton")
        axis.legend(frameon=False, fontsize=7)
    display(fig)
    plt.close(fig)
    return int(best["k"])

## 5. Run the k=5–20 consensus sweep

The diagnostic range now starts at k=5 and extends through k=20. Review PAC,
consensus silhouette, and cluster sizes before explicitly setting `SELECTED_K`.

In [ ]:
W_Z = nodewise_zscore(W_ELIGIBLE.to_numpy())
CONSENSUS_METRICS, CONSENSUS_SOLUTIONS = consensus_sweep(
    W_Z, W_ELIGIBLE.columns.to_numpy()
)
display(CONSENSUS_METRICS.style.format({
    "PAC_0.1_0.9": "{:.6f}",
    "consensus_silhouette": "{:.6f}",
}).background_gradient(subset=["PAC_0.1_0.9"], cmap="Blues_r")
  .background_gradient(subset=["consensus_silhouette"], cmap="Reds"))
DIAGNOSTIC_BEST_K = plot_consensus_diagnostics(CONSENSUS_METRICS)
progress(f"Diagnostic best non-singleton solution: k={DIAGNOSTIC_BEST_K}")
progress("Review the table and figure, then edit SELECTED_K in the next cell.")
ARTIFACTS["consensus_metrics"] = CONSENSUS_METRICS
ARTIFACTS["consensus_solutions"] = CONSENSUS_SOLUTIONS

## 6. Choose the node-group resolution

This notebook intentionally has no default. Enter one integer from 5 through 20
after reviewing the diagnostic table and figure, then continue execution.

In [ ]:
SELECTED_K = 8  # Primary resolution; includes one explicitly qualified singleton program
REFERENCE_K8_PROGRAM_BY_COMPONENT = (0, 5, 5, 7, 6, 0, 7, 1, 7, 7, 3, 3, 4, 0, 3, 0, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 1, 0, 3, 3, 3, 2, 1, 6, 7, 2)

In [ ]:
if SELECTED_K is None:
    raise RuntimeError(
        "PAUSED FOR USER SELECTION: set SELECTED_K to an integer from 5 to 20 "
        "in the cell above, run that cell, then continue from here."
    )

if not isinstance(SELECTED_K, (int, np.integer)) or int(SELECTED_K) not in K_VALUES:
    raise ValueError("SELECTED_K must be an integer from 5 to 20")

SELECTED_K = int(SELECTED_K)
selected_metric = CONSENSUS_METRICS.set_index("k").loc[SELECTED_K]
display(selected_metric.to_frame(name=f"k={SELECTED_K}"))

if not bool(selected_metric["no_singletons"]):
    display(HTML(
        "<div style='padding:10px;background:#fff3cd'><b>Warning:</b> "
        "The selected solution contains a singleton component cluster. "
        "Continuing because singleton solutions are allowed.</div>"
    ))

progress(f"Selected k={SELECTED_K}; starting downstream W/H analysis")

## 7. Materialize the selected component clusters

`CC1…CC8` are groups of the original 50 latent factors derived from W. They are component clusters, not cell identities.

In [ ]:
solution = CONSENSUS_SOLUTIONS[SELECTED_K]
raw_labels = np.asarray(solution["labels"], dtype=int)
consensus = solution["consensus"]
component_names = W_ELIGIBLE.columns.astype(str).to_numpy()
reference_programs = np.asarray(REFERENCE_K8_PROGRAM_BY_COMPONENT, dtype=int)
if len(reference_programs) != len(raw_labels):
    raise RuntimeError("K=8 reference assignment does not match the 50 H/W factors")

overlap = np.zeros((SELECTED_K, SELECTED_K), dtype=int)
for raw_cluster in range(SELECTED_K):
    for program in range(SELECTED_K):
        overlap[raw_cluster, program] = int(np.sum(
            (raw_labels == raw_cluster) & (reference_programs == program)
        ))
raw_rows, program_columns = linear_sum_assignment(-overlap)
raw_to_program = {
    int(raw_cluster): int(program)
    for raw_cluster, program in zip(raw_rows, program_columns)
}
labels = np.asarray([raw_to_program[int(value)] for value in raw_labels], dtype=int)

within = []
for index, label in enumerate(raw_labels):
    peers = np.flatnonzero(raw_labels == label)
    peers = peers[peers != index]
    within.append(float(consensus[index, peers].mean()) if len(peers) else np.nan)
COMPONENT_ASSIGNMENTS = pd.DataFrame({
    "component": component_names,
    "consensus_cluster": labels,
    "within_cluster_consensus": within,
})
show_table(COMPONENT_ASSIGNMENTS, f"Component assignments at k={SELECTED_K}", rows=50)
print("\nCluster sizes:")
display(COMPONENT_ASSIGNMENTS.groupby("consensus_cluster").size().rename("components").to_frame())
ARTIFACTS["component_assignments"] = COMPONENT_ASSIGNMENTS

alignment_rows = []
for raw_cluster, program in sorted(raw_to_program.items(), key=lambda item: item[1]):
    raw_members = raw_labels == raw_cluster
    reference_members = reference_programs == program
    shared = int(np.sum(raw_members & reference_members))
    union = int(np.sum(raw_members | reference_members))
    alignment_rows.append({
        "raw_cluster": raw_cluster,
        "program": program,
        "shared_factors": shared,
        "new_cluster_size": int(raw_members.sum()),
        "reference_cluster_size": int(reference_members.sum()),
        "factor_membership_jaccard": shared / union if union else np.nan,
    })
PROGRAM_ALIGNMENT = pd.DataFrame(alignment_rows)
if (
    set(PROGRAM_ALIGNMENT["program"]) != set(range(SELECTED_K))
    or not PROGRAM_ALIGNMENT["factor_membership_jaccard"].ge(0.5).all()
):
    raise RuntimeError("K=8 consensus clusters do not align to the reviewed program reference")
display(PROGRAM_ALIGNMENT)
ARTIFACTS["program_alignment"] = PROGRAM_ALIGNMENT

## 8. Identify the biological node groups

For each node and component group, enrichment is the mean node-wise W z-score inside that group minus the mean outside it. Each node is retained only for its strongest positive group, and the top 20 unique nodes define each program `N0…N(k−1)`.

### Canonical exponential-decay mixed-node heatmap

Nodes are assigned to their strongest positive-enrichment program, ranked within program × node kind, and weighted as $\exp[-(r-1)/5]$. The smallest prefix reaching 95% cumulative decay mass is retained before deterministic globally unique backfilling to 20 nodes per program.


In [ ]:
from matplotlib.patches import Patch

def score_nodes(node_zscores, metadata, assignments):
    labels_local = assignments["consensus_cluster"].to_numpy()
    records = []
    for cluster_id in range(SELECTED_K):
        inside = labels_local == cluster_id
        outside = ~inside
        frame = metadata.copy()
        frame["consensus_cluster"] = cluster_id
        frame["n_cluster_components"] = int(inside.sum())
        frame["mean_z_inside"] = node_zscores[:, inside].mean(axis=1)
        frame["mean_z_outside"] = node_zscores[:, outside].mean(axis=1)
        frame["enrichment_delta_z"] = (
            frame["mean_z_inside"] - frame["mean_z_outside"]
        )
        frame["category_percentile"] = frame.groupby(
            "node_kind"
        )["enrichment_delta_z"].rank(method="average", pct=True)
        records.append(frame)
    return pd.concat(records, ignore_index=True)


def assign_nodes_to_strongest_cluster(scores):
    """Assign each node once to its strongest positive-enrichment cluster."""
    ordered = scores.sort_values(
        [
            "node_name",
            "enrichment_delta_z",
            "mean_z_inside",
            "consensus_cluster",
        ],
        ascending=[True, False, False, True],
        kind="mergesort",
    )
    assigned = ordered.drop_duplicates("node_name", keep="first").copy()
    assigned = assigned[assigned["enrichment_delta_z"] > 0].copy()
    return assigned.sort_values(
        ["consensus_cluster", "node_kind", "node_name"],
        kind="mergesort",
    ).reset_index(drop=True)


def rank_nodes_with_exponential_decay(
    assigned,
    tau=DECAY_TAU,
    cumulative_mass=DECAY_CUMULATIVE_MASS,
    max_per_group=MAX_NODES_PER_CLUSTER_KIND,
):
    """Rank independently within cluster × node kind and select a decay prefix."""
    if tau <= 0:
        raise ValueError("DECAY_TAU must be positive")
    if not 0 < cumulative_mass <= 1:
        raise ValueError("DECAY_CUMULATIVE_MASS must be in (0, 1]")
    if int(max_per_group) < 1:
        raise ValueError("MAX_NODES_PER_CLUSTER_KIND must be positive")

    ranked_parts = []
    summary_records = []
    group_columns = ["consensus_cluster", "node_kind"]
    for (cluster_id, node_kind), group in assigned.groupby(
        group_columns, sort=True, observed=True
    ):
        group = group.sort_values(
            ["enrichment_delta_z", "mean_z_inside", "node_name"],
            ascending=[False, False, True],
            kind="mergesort",
        ).reset_index(drop=True)
        group["category_rank"] = np.arange(1, len(group) + 1)
        group["decay_weight"] = np.exp(
            -(group["category_rank"].to_numpy(dtype=float) - 1.0) / tau
        )
        total_weight = float(group["decay_weight"].sum())
        group["decay_mass_fraction"] = group["decay_weight"] / total_weight
        group["decay_cumulative_mass"] = group[
            "decay_mass_fraction"
        ].cumsum()

        reached = np.flatnonzero(
            group["decay_cumulative_mass"].to_numpy() >= cumulative_mass
        )
        uncapped_cutoff = int(reached[0] + 1) if len(reached) else len(group)
        cutoff_rank = min(uncapped_cutoff, int(max_per_group))
        group["decay_cutoff_rank"] = cutoff_rank
        group["category_quota"] = cutoff_rank
        group["decay_selected"] = group["category_rank"] <= cutoff_rank
        group["decay_backfill"] = False
        group["selection_reason"] = np.where(
            group["decay_selected"], "decay_prefix", "not_selected"
        )
        achieved_mass = float(
            group.loc[cutoff_rank - 1, "decay_cumulative_mass"]
        )
        summary_records.append({
            "consensus_cluster": int(cluster_id),
            "node_kind": str(node_kind),
            "eligible_positive": int(len(group)),
            "decay_selected": int(cutoff_rank),
            "decay_tau": float(tau),
            "target_cumulative_mass": float(cumulative_mass),
            "achieved_cumulative_mass": achieved_mass,
            "maximum_per_cluster_kind": int(max_per_group),
            "cap_reached": bool(uncapped_cutoff > max_per_group),
            "target_reached": bool(achieved_mass >= cumulative_mass),
        })
        ranked_parts.append(group)

    ranked = pd.concat(ranked_parts, ignore_index=True)
    ranked = ranked.sort_values(
        ["consensus_cluster", "node_kind", "category_rank"],
        kind="mergesort",
    ).reset_index(drop=True)
    summary = pd.DataFrame(summary_records).sort_values(
        ["consensus_cluster", "node_kind"], kind="mergesort"
    ).reset_index(drop=True)
    return ranked, summary


def select_mixed_decay_budget(
    assigned_ranked, cluster_local_ranked, per_cluster_budget, selected_k
):
    """Fill a globally unique fixed budget, prioritizing strongest assignments."""
    selected_parts = []
    used_nodes = set()
    priority_columns = [
        "decay_weight",
        "enrichment_delta_z",
        "mean_z_inside",
        "node_name",
    ]
    priority_ascending = [False, False, False, True]

    assigned_prefix_counts = (
        assigned_ranked[assigned_ranked["decay_selected"]]
        .groupby("consensus_cluster")
        .size()
        .reindex(range(selected_k), fill_value=0)
    )
    cluster_order = assigned_prefix_counts.sort_values(
        ascending=True, kind="mergesort"
    ).index.tolist()

    for cluster_id in cluster_order:
        assigned_rows = assigned_ranked[
            assigned_ranked["consensus_cluster"].eq(cluster_id)
            & assigned_ranked["decay_selected"]
            & ~assigned_ranked["node_name"].isin(used_nodes)
        ].sort_values(
            priority_columns,
            ascending=priority_ascending,
            kind="mergesort",
        )
        chosen = assigned_rows.head(per_cluster_budget).copy()
        chosen["decay_backfill"] = False
        chosen["selection_reason"] = "strongest_cluster_decay_prefix"

        if len(chosen) < per_cluster_budget:
            local_rows = cluster_local_ranked[
                cluster_local_ranked["consensus_cluster"].eq(cluster_id)
                & cluster_local_ranked["decay_selected"]
                & ~cluster_local_ranked["node_name"].isin(used_nodes)
                & ~cluster_local_ranked["node_name"].isin(
                    chosen["node_name"]
                )
            ].sort_values(
                priority_columns,
                ascending=priority_ascending,
                kind="mergesort",
            )
            local_fill = local_rows.head(
                per_cluster_budget - len(chosen)
            ).copy()
            local_fill["decay_backfill"] = True
            local_fill["selection_reason"] = "cluster_local_decay_prefix"
            chosen = pd.concat([chosen, local_fill], ignore_index=True)

        if len(chosen) < per_cluster_budget:
            remaining = cluster_local_ranked[
                cluster_local_ranked["consensus_cluster"].eq(cluster_id)
                & ~cluster_local_ranked["node_name"].isin(used_nodes)
                & ~cluster_local_ranked["node_name"].isin(
                    chosen["node_name"]
                )
            ].sort_values(
                priority_columns,
                ascending=priority_ascending,
                kind="mergesort",
            )
            backfill = remaining.head(
                per_cluster_budget - len(chosen)
            ).copy()
            backfill["decay_backfill"] = True
            backfill["selection_reason"] = "budget_backfill"
            chosen = pd.concat([chosen, backfill], ignore_index=True)

        if len(chosen) != per_cluster_budget:
            raise RuntimeError(
                f"Cluster {cluster_id} has only {len(chosen)} available "
                f"unique positive nodes for a {per_cluster_budget}-node budget"
            )
        chosen = chosen.sort_values(
            priority_columns,
            ascending=priority_ascending,
            kind="mergesort",
        ).reset_index(drop=True)
        chosen["cluster_rank"] = np.arange(1, len(chosen) + 1)
        used_nodes.update(chosen["node_name"].tolist())
        selected_parts.append(chosen)

    return pd.concat(selected_parts, ignore_index=True).sort_values(
        ["consensus_cluster", "cluster_rank"], kind="mergesort"
    ).reset_index(drop=True)


def plot_top_enriched_heatmap(selected, node_zscores, assignments):
    lookup = {
        name: index
        for index, name in enumerate(NODE_METADATA_ELIGIBLE["node_name"])
    }
    row_positions = np.asarray([
        lookup[name] for name in selected["node_name"]
    ])
    component_lookup = assignments.set_index(
        "component"
    )["consensus_cluster"]
    component_labels = component_lookup.loc[
        W_ELIGIBLE.columns.astype(str)
    ].to_numpy()
    column_order = np.lexsort((
        W_ELIGIBLE.columns.astype(int),
        component_labels,
    ))
    matrix = node_zscores[row_positions][:, column_order]
    component_labels = component_labels[column_order]
    ordered_names = (
        W_ELIGIBLE.columns.astype(str).to_numpy()[column_order]
    )
    row_clusters = selected["consensus_cluster"].to_numpy()
    categories = selected["node_kind"].astype(str).to_numpy()
    limit = max(float(np.quantile(np.abs(matrix), 0.99)), 1.0)

    height = max(10, 0.18 * len(selected))
    figure = plt.figure(figsize=(16, height), constrained_layout=True)
    grid = figure.add_gridspec(
        2, 4,
        width_ratios=(0.20, 0.20, 12, 2.8),
        height_ratios=(0.35, 12),
    )
    top_axis = figure.add_subplot(grid[0, 2])
    program_axis = figure.add_subplot(grid[1, 0])
    kind_axis = figure.add_subplot(grid[1, 1])
    axis = figure.add_subplot(grid[1, 2])
    legend_axis = figure.add_subplot(grid[:, 3])
    legend_axis.axis("off")

    image = axis.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-limit,
        vmax=limit,
        rasterized=True,
    )
    for boundary in np.flatnonzero(np.diff(row_clusters) != 0) + 0.5:
        axis.axhline(boundary, color="#222222", linewidth=0.8)
        program_axis.axhline(boundary, color="#222222", linewidth=0.8)
        kind_axis.axhline(boundary, color="#222222", linewidth=0.8)
    for boundary in (
        np.flatnonzero(np.diff(component_labels) != 0) + 0.5
    ):
        axis.axvline(boundary, color="#222222", linewidth=0.8)
        top_axis.axvline(boundary, color="#222222", linewidth=0.8)

    axis.set_xticks(np.arange(len(ordered_names)))
    axis.set_xticklabels(ordered_names, rotation=90, fontsize=6)
    axis.set_yticks(np.arange(len(selected)))
    axis.set_yticklabels(selected["node_name"], fontsize=5)
    axis.set_xlabel("Components ordered by selected consensus cluster")
    axis.set_ylabel("Exponential-decay mixed nodes grouped by program")
    axis.set_title(
        f"Exponential-decay enriched nodes per component cluster "
        f"(k={SELECTED_K})"
    )

    top_axis.imshow(
        component_labels[None, :],
        aspect="auto",
        interpolation="nearest",
        cmap=ListedColormap(CLUSTER_COLORS[:SELECTED_K]),
        vmin=0,
        vmax=max(SELECTED_K - 1, 1),
    )
    program_axis.imshow(
        row_clusters[:, None],
        aspect="auto",
        interpolation="nearest",
        cmap=ListedColormap(CLUSTER_COLORS[:SELECTED_K]),
        vmin=0,
        vmax=max(SELECTED_K - 1, 1),
    )
    category_codes = np.asarray([
        list(CATEGORY_COLORS).index(value)
        if value in CATEGORY_COLORS else 0
        for value in categories
    ])
    kind_axis.imshow(
        category_codes[:, None],
        aspect="auto",
        interpolation="nearest",
        cmap=ListedColormap(list(CATEGORY_COLORS.values())),
        vmin=0,
        vmax=max(len(CATEGORY_COLORS) - 1, 1),
    )
    for annotation_axis, label in [
        (top_axis, "Component cluster"),
        (program_axis, "Node group"),
        (kind_axis, "Node kind"),
    ]:
        annotation_axis.set_xticks([])
        annotation_axis.set_yticks([])
        annotation_axis.set_title(label, fontsize=7)
        for spine in annotation_axis.spines.values():
            spine.set_visible(False)

    cluster_legend = legend_axis.legend(
        handles=[
            Patch(
                facecolor=CLUSTER_COLORS[index],
                edgecolor="none",
                label=f"FG{index + 1}",
            )
            for index in range(SELECTED_K)
        ],
        title="Component cluster",
        loc="upper left",
        frameon=False,
        fontsize=6.5,
        title_fontsize=7.5,
    )
    legend_axis.add_artist(cluster_legend)
    legend_axis.legend(
        handles=[
            Patch(
                facecolor=color, edgecolor="none", label=kind
            )
            for kind, color in CATEGORY_COLORS.items()
            if kind in set(categories)
        ],
        title="Node kind",
        loc="lower left",
        frameon=False,
        fontsize=6.5,
        title_fontsize=7.5,
    )
    figure.colorbar(
        image,
        ax=axis,
        fraction=0.025,
        pad=0.015,
        label="Node-wise z-score of log(1 + W)",
    )
    display(figure)
    plt.close(figure)


progress("Scoring all nonzero W nodes against the selected component clusters")
ALL_NODE_SCORES = score_nodes(
    W_Z, NODE_METADATA_ELIGIBLE, COMPONENT_ASSIGNMENTS
)
ASSIGNED_POSITIVE_NODES = assign_nodes_to_strongest_cluster(
    ALL_NODE_SCORES
)
DECAY_RANKED_NODES, DECAY_SELECTION_SUMMARY = (
    rank_nodes_with_exponential_decay(ASSIGNED_POSITIVE_NODES)
)
DECAY_SELECTED_NODES = DECAY_RANKED_NODES[
    DECAY_RANKED_NODES["decay_selected"]
].copy()
CLUSTER_LOCAL_POSITIVE_NODES = ALL_NODE_SCORES[
    ALL_NODE_SCORES["enrichment_delta_z"] > 0
].copy()
LOCAL_DECAY_RANKED_NODES, LOCAL_DECAY_SELECTION_SUMMARY = (
    rank_nodes_with_exponential_decay(CLUSTER_LOCAL_POSITIVE_NODES)
)
MIXED_TOP_NODES = select_mixed_decay_budget(
    DECAY_RANKED_NODES,
    LOCAL_DECAY_RANKED_NODES,
    TOP_N,
    SELECTED_K,
)
repeat_mixed = select_mixed_decay_budget(
    DECAY_RANKED_NODES,
    LOCAL_DECAY_RANKED_NODES,
    TOP_N,
    SELECTED_K,
)
DECAY_DETERMINISTIC = MIXED_TOP_NODES[
    ["consensus_cluster", "node_name", "cluster_rank"]
].equals(
    repeat_mixed[
        ["consensus_cluster", "node_name", "cluster_rank"]
    ]
)

# Compatibility aliases retained for downstream notebook consumers.
UNCONSTRAINED_TOP_NODES = DECAY_SELECTED_NODES
MIXED_CATEGORY_ALLOCATION = DECAY_SELECTION_SUMMARY

show_table(
    DECAY_SELECTION_SUMMARY,
    (
        "Exponential-decay selection audit by component cluster and "
        "node kind"
    ),
    rows=40,
)
show_table(
    MIXED_TOP_NODES,
    f"Decay-ranked canonical mixed nodes at k={SELECTED_K}",
    rows=30,
)
mixed_counts = pd.crosstab(
    MIXED_TOP_NODES["consensus_cluster"],
    MIXED_TOP_NODES["node_kind"],
)
display(HTML(
    "<h4>Exponential-decay node-kind composition of the canonical mixed set</h4>"
))
display(mixed_counts)

expected_mixed_rows = SELECTED_K * TOP_N
DECAY_FORMULA_CHECK = all(
    np.allclose(
        table["decay_weight"].to_numpy(),
        np.exp(
            -(
                table["category_rank"].to_numpy(dtype=float) - 1.0
            ) / DECAY_TAU
        ),
    )
    for table in [DECAY_RANKED_NODES, LOCAL_DECAY_RANKED_NODES]
)
DECAY_PREFIX_CHECK = (
    DECAY_RANKED_NODES["decay_selected"].to_numpy()
    == (
        DECAY_RANKED_NODES["category_rank"].to_numpy()
        <= DECAY_RANKED_NODES["decay_cutoff_rank"].to_numpy()
    )
).all()
DECAY_MASS_CHECK = all(
    summary["target_reached"].all()
    for summary in [
        DECAY_SELECTION_SUMMARY, LOCAL_DECAY_SELECTION_SUMMARY
    ]
)
DECAY_CAP_CHECK = all(
    summary["decay_selected"].le(
        MAX_NODES_PER_CLUSTER_KIND
    ).all()
    for summary in [
        DECAY_SELECTION_SUMMARY, LOCAL_DECAY_SELECTION_SUMMARY
    ]
)
within_cluster_unique = (
    MIXED_TOP_NODES.groupby(
        ["consensus_cluster", "node_name"]
    ).size().max()
    == 1
)
if not (
    len(MIXED_TOP_NODES) == expected_mixed_rows
    and MIXED_TOP_NODES.groupby(
        "consensus_cluster"
    ).size().eq(TOP_N).all()
    and MIXED_TOP_NODES["enrichment_delta_z"].gt(0).all()
    and MIXED_TOP_NODES["node_name"].is_unique
    and within_cluster_unique
    and DECAY_FORMULA_CHECK
    and DECAY_PREFIX_CHECK
    and DECAY_MASS_CHECK
    and DECAY_CAP_CHECK
    and DECAY_DETERMINISTIC
):
    raise RuntimeError("Exponential-decay node selection validation failed")

plot_top_enriched_heatmap(
    MIXED_TOP_NODES, W_Z, COMPONENT_ASSIGNMENTS
)
ARTIFACTS["all_node_scores"] = ALL_NODE_SCORES
ARTIFACTS["unconstrained_top_nodes"] = UNCONSTRAINED_TOP_NODES
ARTIFACTS["mixed_top_nodes"] = MIXED_TOP_NODES
ARTIFACTS["mixed_category_allocation"] = MIXED_CATEGORY_ALLOCATION
ARTIFACTS["decay_ranked_nodes"] = DECAY_RANKED_NODES
ARTIFACTS["cluster_local_decay_ranked_nodes"] = (
    LOCAL_DECAY_RANKED_NODES
)
ARTIFACTS["cluster_local_decay_selection_summary"] = (
    LOCAL_DECAY_SELECTION_SUMMARY
)
ARTIFACTS["decay_selected_nodes"] = DECAY_SELECTED_NODES
ARTIFACTS["decay_selection_summary"] = DECAY_SELECTION_SUMMARY
ARTIFACTS["decay_parameters"] = {
    "tau": DECAY_TAU,
    "cumulative_mass": DECAY_CUMULATIVE_MASS,
    "maximum_per_cluster_kind": MAX_NODES_PER_CLUSTER_KIND,
    "main_panel_nodes_per_cluster": TOP_N,
}
progress(
    f"Selected {len(MIXED_TOP_NODES):,} exponential-decay mixed nodes "
    f"({TOP_N} per cluster)"
)


## 8b. Relative W enrichment and node-kind composition

Raw component-cluster means are retained for validation, while the displayed
views focus on relative enrichment and its node-kind composition.

In [ ]:
# W_COMPONENT_CLUSTER_MEANS_V2
progress("Averaging W weights across the selected component clusters")

component_to_cluster = (
    COMPONENT_ASSIGNMENTS
    .set_index("component")["consensus_cluster"]
    .astype(int)
)
expected_components = W_ELIGIBLE.columns.astype(str)
assigned_components = component_to_cluster.index.astype(str)

if (
    len(component_to_cluster) != len(expected_components)
    or not component_to_cluster.index.is_unique
    or set(assigned_components) != set(expected_components)
):
    raise RuntimeError(
        "Component assignments must contain each W component exactly once"
    )

W_BY_COMPONENT_MEAN = pd.DataFrame(
    {
        f"FG{cluster + 1}": W_ELIGIBLE.loc[
            :,
            component_to_cluster.index[
                component_to_cluster.eq(cluster)
            ].astype(str),
        ].mean(axis=1)
        for cluster in range(SELECTED_K)
    },
    index=W_ELIGIBLE.index,
)
W_BY_COMPONENT_MEAN.index.name = (
    W_ELIGIBLE.index.name or "node_name"
)
W_BY_COMPONENT_LOG_ZSCORE = pd.DataFrame(
    nodewise_zscore(W_BY_COMPONENT_MEAN.to_numpy()),
    index=W_BY_COMPONENT_MEAN.index,
    columns=W_BY_COMPONENT_MEAN.columns,
)
W_BY_COMPONENT_LOG_ZSCORE.index.name = (
    W_BY_COMPONENT_MEAN.index.name
)

show_table(
    W_BY_COMPONENT_LOG_ZSCORE,
    "Node-wise log/z-scored mean W enrichment by shared component cluster",
    rows=20,
)

selected_node_names = MIXED_TOP_NODES["node_name"].astype(str).tolist()
selected_w_z = W_BY_COMPONENT_LOG_ZSCORE.loc[
    selected_node_names
].to_numpy(dtype=float)
selected_programs = (
    MIXED_TOP_NODES["consensus_cluster"].to_numpy()
)
selected_kinds = MIXED_TOP_NODES["node_kind"].astype(str).to_numpy()
z_limit = max(
    float(np.nanpercentile(np.abs(selected_w_z), 99)),
    1.0,
)
figure_height = max(9.0, 0.16 * len(selected_node_names))
figure = plt.figure(
    figsize=(14, figure_height), constrained_layout=True
)
grid = figure.add_gridspec(
    1, 4, width_ratios=(0.20, 0.20, 10, 2.6)
)
program_axis = figure.add_subplot(grid[0, 0])
kind_axis = figure.add_subplot(grid[0, 1])
axis = figure.add_subplot(grid[0, 2])
legend_axis = figure.add_subplot(grid[0, 3])
legend_axis.axis("off")

image = axis.imshow(
    selected_w_z,
    aspect="auto",
    interpolation="nearest",
    cmap="RdBu_r",
    vmin=-z_limit,
    vmax=z_limit,
    rasterized=True,
)
boundaries = (
    np.flatnonzero(np.diff(selected_programs) != 0) + 0.5
)
for panel in [program_axis, kind_axis, axis]:
    for boundary in boundaries:
        panel.axhline(boundary, color="#222222", linewidth=0.8)

program_axis.imshow(
    selected_programs[:, None],
    aspect="auto",
    interpolation="nearest",
    cmap=ListedColormap(CLUSTER_COLORS[:SELECTED_K]),
    vmin=0,
    vmax=max(SELECTED_K - 1, 1),
)
kind_codes = np.asarray([
    list(CATEGORY_COLORS).index(kind) for kind in selected_kinds
])
kind_axis.imshow(
    kind_codes[:, None],
    aspect="auto",
    interpolation="nearest",
    cmap=ListedColormap(list(CATEGORY_COLORS.values())),
    vmin=0,
    vmax=max(len(CATEGORY_COLORS) - 1, 1),
)
for annotation_axis, label in [
    (program_axis, "Node group"),
    (kind_axis, "Node kind"),
]:
    annotation_axis.set_xticks([])
    annotation_axis.set_yticks([])
    annotation_axis.set_title(label, fontsize=7)
    for spine in annotation_axis.spines.values():
        spine.set_visible(False)

axis.set_xticks(range(SELECTED_K))
axis.set_xticklabels(
    W_BY_COMPONENT_LOG_ZSCORE.columns,
    rotation=45,
    ha="right",
)
axis.set_yticks(range(len(selected_node_names)))
axis.set_yticklabels(selected_node_names, fontsize=5)
axis.set_xlabel("Component cluster")
axis.set_ylabel("Canonical mixed nodes ordered by program and rank")
axis.set_title(
    "Relative W enrichment of exponential-decay mixed nodes",
    loc="left",
    fontweight="bold",
)
cluster_legend = legend_axis.legend(
    handles=[
        Patch(
            facecolor=CLUSTER_COLORS[index],
            edgecolor="none",
            label=f"FG{index + 1}",
        )
        for index in range(SELECTED_K)
    ],
    title="Program cluster",
    loc="upper left",
    frameon=False,
    fontsize=6.5,
    title_fontsize=7.5,
)
legend_axis.add_artist(cluster_legend)
legend_axis.legend(
    handles=[
        Patch(
            facecolor=color, edgecolor="none", label=kind
        )
        for kind, color in CATEGORY_COLORS.items()
        if kind in set(selected_kinds)
    ],
    title="Node kind",
    loc="lower left",
    frameon=False,
    fontsize=6.5,
    title_fontsize=7.5,
)
figure.colorbar(
    image,
    ax=axis,
    fraction=0.035,
    pad=0.02,
    label="Node-wise z-score of log(1 + mean W)",
)
display(figure)
plt.close(figure)

# Four complementary views of mixed-node enrichment composition.
composition_grouped = MIXED_TOP_NODES.groupby(
    ["consensus_cluster", "node_kind"]
)["enrichment_delta_z"]

ENRICHMENT_SUM_BY_CLUSTER_KIND = (
    composition_grouped.sum()
    .unstack(fill_value=0)
    .reindex(index=range(SELECTED_K), fill_value=0)
)
SELECTED_COUNT_BY_CLUSTER_KIND = (
    composition_grouped.size()
    .unstack(fill_value=0)
    .reindex(index=range(SELECTED_K), fill_value=0)
    .astype(int)
)
MEAN_ENRICHMENT_BY_CLUSTER_KIND = (
    composition_grouped.mean()
    .unstack()
    .reindex(index=range(SELECTED_K))
)

kind_order = [
    kind for kind in CATEGORY_COLORS
    if kind in ENRICHMENT_SUM_BY_CLUSTER_KIND.columns
]
for table_name in [
    "ENRICHMENT_SUM_BY_CLUSTER_KIND",
    "SELECTED_COUNT_BY_CLUSTER_KIND",
    "MEAN_ENRICHMENT_BY_CLUSTER_KIND",
]:
    globals()[table_name] = globals()[table_name].reindex(
        columns=kind_order
    )

ENRICHMENT_PERCENT_BY_CLUSTER_KIND = (
    ENRICHMENT_SUM_BY_CLUSTER_KIND.div(
        ENRICHMENT_SUM_BY_CLUSTER_KIND.sum(axis=1),
        axis=0,
    ) * 100
)


def draw_stacked_composition(
    axis, table, ylabel, title, percent=False
):
    bottom = np.zeros(SELECTED_K, dtype=float)
    x_positions = np.arange(SELECTED_K)
    for kind in kind_order:
        values = table[kind].to_numpy(dtype=float)
        axis.bar(
            x_positions,
            values,
            bottom=bottom,
            color=CATEGORY_COLORS[kind],
            edgecolor="white",
            linewidth=0.35,
            label=kind,
        )
        bottom += values
    axis.set_xticks(
        x_positions,
        [f"FG{index + 1}" for index in range(SELECTED_K)],
        rotation=45 if SELECTED_K > 10 else 0,
        ha="right" if SELECTED_K > 10 else "center",
    )
    axis.set_xlabel("Component cluster")
    axis.set_ylabel(ylabel)
    axis.set_title(title, loc="left", fontweight="bold")
    if percent:
        axis.set_ylim(0, 100)


composition_width = max(14.0, 0.85 * SELECTED_K)
fig_composition, composition_axes = plt.subplots(
    2,
    2,
    figsize=(composition_width, 10.0),
    constrained_layout=True,
)
draw_stacked_composition(
    composition_axes[0, 0],
    ENRICHMENT_SUM_BY_CLUSTER_KIND,
    "Summed positive enrichment Δz",
    "A  Absolute enrichment contribution",
)
draw_stacked_composition(
    composition_axes[0, 1],
    ENRICHMENT_PERCENT_BY_CLUSTER_KIND,
    "Share of total enrichment (%)",
    "B  Relative enrichment composition",
    percent=True,
)
draw_stacked_composition(
    composition_axes[1, 0],
    SELECTED_COUNT_BY_CLUSTER_KIND,
    "Selected mixed nodes",
    "C  Exponential-decay decay-ranked composition",
)

dot_axis = composition_axes[1, 1]
mean_values = MEAN_ENRICHMENT_BY_CLUSTER_KIND.to_numpy(
    dtype=float
)
count_values = SELECTED_COUNT_BY_CLUSTER_KIND.to_numpy(
    dtype=float
)
present = count_values > 0
dot_x, dot_y = np.meshgrid(
    np.arange(SELECTED_K), np.arange(len(kind_order)),
    indexing="ij",
)
maximum_count = max(float(count_values.max()), 1.0)
dot_sizes = 35 + 260 * count_values[present] / maximum_count
dot_color_min = float(np.nanmin(mean_values[present]))
dot_color_max = float(np.nanmax(mean_values[present]))
if np.isclose(dot_color_min, dot_color_max):
    dot_color_max = dot_color_min + 1e-12
dot_norm = matplotlib.colors.Normalize(
    vmin=dot_color_min, vmax=dot_color_max
)
dot_image = dot_axis.scatter(
    dot_x[present],
    dot_y[present],
    s=dot_sizes,
    c=mean_values[present],
    cmap="viridis",
    norm=dot_norm,
    edgecolor="white",
    linewidth=0.45,
)
dot_axis.set_xticks(
    range(SELECTED_K),
    [f"FG{index + 1}" for index in range(SELECTED_K)],
    rotation=45 if SELECTED_K > 10 else 0,
    ha="right" if SELECTED_K > 10 else "center",
)
dot_axis.set_yticks(range(len(kind_order)), kind_order)
dot_axis.set_xlabel("Component cluster")
dot_axis.set_ylabel("Node kind")
dot_axis.set_title(
    "D  Mean enrichment and selected-node count",
    loc="left",
    fontweight="bold",
)
fig_composition.colorbar(
    dot_image,
    ax=dot_axis,
    fraction=0.045,
    pad=0.03,
    label="Mean positive enrichment Δz",
)
count_references = sorted(set([
    1,
    max(1, int(np.ceil(maximum_count / 2))),
    int(maximum_count),
]))
size_handles = [
    dot_axis.scatter(
        [], [],
        s=35 + 260 * value / maximum_count,
        facecolor="#777777",
        edgecolor="white",
        label=str(value),
    )
    for value in count_references
]
size_legend = dot_axis.legend(
    handles=size_handles,
    title="Selected nodes",
    loc="upper left",
    bbox_to_anchor=(1.16, 0.98),
    frameon=False,
    fontsize=7,
    title_fontsize=8,
)
dot_axis.add_artist(size_legend)

legend_handles, legend_labels = (
    composition_axes[0, 0].get_legend_handles_labels()
)
fig_composition.legend(
    legend_handles,
    legend_labels,
    title="Node kind",
    loc="outside right center",
    frameon=False,
    fontsize=7,
    title_fontsize=8,
)
fig_composition.suptitle(
    "Exponential-decay mixed-node enrichment composition",
    fontsize=13,
    fontweight="bold",
)
display(fig_composition)
plt.close(fig_composition)

component_cluster_sizes = (
    component_to_cluster.value_counts().sort_index()
)
direct_means_match = all(
    np.allclose(
        W_BY_COMPONENT_MEAN[f"FG{cluster + 1}"].to_numpy(),
        W_ELIGIBLE.loc[
            :,
            component_to_cluster.index[
                component_to_cluster.eq(cluster)
            ].astype(str),
        ].mean(axis=1).to_numpy(),
        rtol=1e-12,
        atol=1e-12,
    )
    for cluster in range(SELECTED_K)
)
W_COMPONENT_MEAN_CHECKS = pd.Series({
    "shape_nonzero_nodes_by_selected_k":
        W_BY_COMPONENT_MEAN.shape
        == (len(W_ELIGIBLE), SELECTED_K),
    "columns_FG1_to_FGk":
        W_BY_COMPONENT_MEAN.columns.tolist()
        == [f"FG{cluster + 1}" for cluster in range(SELECTED_K)],
    "index_matches_nonzero_W":
        W_BY_COMPONENT_MEAN.index.equals(W_ELIGIBLE.index),
    "all_components_assigned_once":
        len(component_to_cluster) == W_ELIGIBLE.shape[1]
        and component_to_cluster.index.is_unique
        and set(assigned_components) == set(expected_components),
    "no_empty_component_clusters":
        component_cluster_sizes.reindex(
            range(SELECTED_K), fill_value=0
        ).gt(0).all(),
    "raw_means_match_direct_calculation": direct_means_match,
    "raw_means_nonnegative":
        W_BY_COMPONENT_MEAN.to_numpy().min() >= 0,
    "raw_means_finite":
        np.isfinite(W_BY_COMPONENT_MEAN.to_numpy()).all(),
    "normalized_means_finite":
        np.isfinite(W_BY_COMPONENT_LOG_ZSCORE.to_numpy()).all(),
    "heatmap_rows_match_mixed_nodes":
        selected_node_names
        == MIXED_TOP_NODES["node_name"].astype(str).tolist(),
    "enrichment_sum_nonnegative":
        ENRICHMENT_SUM_BY_CLUSTER_KIND.to_numpy().min() >= 0,
    "enrichment_sum_finite":
        np.isfinite(
            ENRICHMENT_SUM_BY_CLUSTER_KIND.to_numpy()
        ).all(),
    "enrichment_percent_sums_100":
        np.allclose(
            ENRICHMENT_PERCENT_BY_CLUSTER_KIND.sum(axis=1),
            100.0,
        ),
    "selected_counts_sum_TOP_N":
        SELECTED_COUNT_BY_CLUSTER_KIND.sum(axis=1).eq(TOP_N).all(),
    "mean_enrichment_positive_where_selected":
        (
            MEAN_ENRICHMENT_BY_CLUSTER_KIND.where(
                SELECTED_COUNT_BY_CLUSTER_KIND.gt(0)
            ).stack().gt(0).all()
        ),
}, name="passed")
display(W_COMPONENT_MEAN_CHECKS.to_frame())
if not W_COMPONENT_MEAN_CHECKS.all():
    raise RuntimeError(
        "Mean W-by-component-cluster validation failed"
    )

ARTIFACTS["W_by_component_mean"] = W_BY_COMPONENT_MEAN
ARTIFACTS[
    "W_by_component_log_zscore"
] = W_BY_COMPONENT_LOG_ZSCORE
ARTIFACTS[
    "W_component_mean_checks"
] = W_COMPONENT_MEAN_CHECKS
ARTIFACTS[
    "enrichment_sum_by_cluster_kind"
] = ENRICHMENT_SUM_BY_CLUSTER_KIND
ARTIFACTS[
    "enrichment_percent_by_cluster_kind"
] = ENRICHMENT_PERCENT_BY_CLUSTER_KIND
ARTIFACTS[
    "selected_count_by_cluster_kind"
] = SELECTED_COUNT_BY_CLUSTER_KIND
ARTIFACTS[
    "mean_enrichment_by_cluster_kind"
] = MEAN_ENRICHMENT_BY_CLUSTER_KIND
progress(
    "Relative W heatmap and four-view enrichment composition complete"
)

## 9. Category-specific node-group views

The same enrichment calculation is repeated separately for every node category. These category-specific selections are exploratory views and are separate from the global top-node set used for reconstruction.

In [ ]:
def plot_category_specific_heatmap(category, selected_rows):
    node_lookup = {
        name: index
        for index, name in enumerate(
            NODE_METADATA_ELIGIBLE["node_name"]
        )
    }
    row_positions = np.asarray([
        node_lookup[name] for name in selected_rows["node_name"]
    ])
    component_lookup = COMPONENT_ASSIGNMENTS.set_index(
        "component"
    )["consensus_cluster"]
    component_labels = component_lookup.loc[
        W_ELIGIBLE.columns.astype(str)
    ].to_numpy()
    column_order = np.lexsort((
        W_ELIGIBLE.columns.astype(int),
        component_labels,
    ))
    ordered_component_labels = component_labels[column_order]
    ordered_component_names = (
        W_ELIGIBLE.columns.astype(str).to_numpy()[column_order]
    )
    matrix = W_Z[row_positions][:, column_order]
    row_clusters = selected_rows[
        "consensus_cluster"
    ].to_numpy()
    color_limit = max(
        float(np.quantile(np.abs(matrix), 0.99)), 1.0
    )

    figure_height = max(6.0, 0.19 * len(selected_rows))
    figure = plt.figure(
        figsize=(16, figure_height), constrained_layout=True
    )
    grid = figure.add_gridspec(
        2, 3,
        width_ratios=(0.25, 12, 2.6),
        height_ratios=(0.35, 12),
    )
    top_axis = figure.add_subplot(grid[0, 1])
    side_axis = figure.add_subplot(grid[1, 0])
    axis = figure.add_subplot(grid[1, 1])
    legend_axis = figure.add_subplot(grid[:, 2])
    legend_axis.axis("off")
    image = axis.imshow(
        matrix,
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-color_limit,
        vmax=color_limit,
        rasterized=True,
    )
    for boundary in (
        np.flatnonzero(np.diff(row_clusters) != 0) + 0.5
    ):
        axis.axhline(boundary, color="#222222", linewidth=0.8)
        side_axis.axhline(
            boundary, color="#222222", linewidth=0.8
        )
    for boundary in (
        np.flatnonzero(
            np.diff(ordered_component_labels) != 0
        ) + 0.5
    ):
        axis.axvline(boundary, color="#222222", linewidth=0.8)
        top_axis.axvline(
            boundary, color="#222222", linewidth=0.8
        )

    axis.set_xticks(range(len(ordered_component_names)))
    axis.set_xticklabels(
        ordered_component_names, rotation=90, fontsize=6.5
    )
    axis.set_yticks(range(len(selected_rows)))
    axis.set_yticklabels(
        selected_rows["node_name"], fontsize=5.5
    )
    axis.set_xlabel(
        "Components ordered by selected consensus cluster"
    )
    axis.set_ylabel(f"Exponential-decay selected {category} nodes")
    axis.set_title(
        f"Exponential-decay selected {category} nodes across W components "
        f"(k={SELECTED_K})"
    )
    top_axis.imshow(
        ordered_component_labels[None, :],
        aspect="auto",
        interpolation="nearest",
        cmap=ListedColormap(CLUSTER_COLORS[:SELECTED_K]),
        vmin=0,
        vmax=max(SELECTED_K - 1, 1),
    )
    side_axis.imshow(
        row_clusters[:, None],
        aspect="auto",
        interpolation="nearest",
        cmap=ListedColormap(CLUSTER_COLORS[:SELECTED_K]),
        vmin=0,
        vmax=max(SELECTED_K - 1, 1),
    )
    top_axis.set_title("Component cluster", fontsize=7)
    side_axis.set_title("Node group", fontsize=7)
    for annotation_axis in [top_axis, side_axis]:
        annotation_axis.set_xticks([])
        annotation_axis.set_yticks([])
        for spine in annotation_axis.spines.values():
            spine.set_visible(False)
    legend_axis.legend(
        handles=[
            Patch(
                facecolor=CLUSTER_COLORS[index],
                edgecolor="none",
                label=f"FG{index + 1}",
            )
            for index in range(SELECTED_K)
        ],
        title="Component cluster",
        loc="upper left",
        frameon=False,
        fontsize=6.5,
        title_fontsize=7.5,
    )
    figure.colorbar(
        image,
        ax=axis,
        fraction=0.025,
        pad=0.02,
        label="Node-wise z-score of log(1 + W)",
    )
    display(figure)
    plt.close(figure)


CATEGORY_SPECIFIC_MIXED_TOP_NODES = {}
category_summary = []
for category in NODE_METADATA_ELIGIBLE["node_kind"].drop_duplicates():
    selected_category = DECAY_SELECTED_NODES[
        DECAY_SELECTED_NODES["node_kind"].eq(category)
    ].sort_values(
        ["consensus_cluster", "category_rank"], kind="mergesort"
    ).reset_index(drop=True)
    if selected_category.empty:
        progress(
            f"Skipping {category}: no positively enriched nodes"
        )
        continue
    CATEGORY_SPECIFIC_MIXED_TOP_NODES[
        str(category)
    ] = selected_category
    per_cluster_counts = selected_category.groupby(
        "consensus_cluster"
    ).size()
    category_summary.append({
        "node_kind": str(category),
        "selected_nodes": len(selected_category),
        "represented_clusters": selected_category[
            "consensus_cluster"
        ].nunique(),
        "maximum_selected_in_one_cluster": int(per_cluster_counts.max()),
        "decay_tau": DECAY_TAU,
        "target_cumulative_mass": DECAY_CUMULATIVE_MASS,
    })
    progress(
        f"Displaying exponential-decay heatmap: {category} "
        f"({len(selected_category)} nodes)"
    )
    plot_category_specific_heatmap(
        str(category), selected_category
    )

CATEGORY_HEATMAP_SUMMARY = pd.DataFrame(category_summary)
display(CATEGORY_HEATMAP_SUMMARY)
category_union = pd.concat(
    CATEGORY_SPECIFIC_MIXED_TOP_NODES.values(), ignore_index=True
)
CATEGORY_DECAY_SELECTION_CHECK = (
    set(category_union["node_name"])
    == set(DECAY_SELECTED_NODES["node_name"])
    and category_union["node_name"].is_unique
    and category_union.groupby(
        ["consensus_cluster", "node_kind"]
    ).size().le(MAX_NODES_PER_CLUSTER_KIND).all()
)
if not CATEGORY_DECAY_SELECTION_CHECK:
    raise RuntimeError(
        "Category-specific panels do not match decay-selected nodes"
    )
ARTIFACTS[
    "category_specific_top_nodes"
] = CATEGORY_SPECIFIC_MIXED_TOP_NODES
ARTIFACTS[
    "category_heatmap_summary"
] = CATEGORY_HEATMAP_SUMMARY


## 9b. K=6 versus K=8 sensitivity

Retain the validated stability comparison while using K=8 as the primary biological resolution.

In [ ]:
# Preserve the validated K=6 versus K=8 sensitivity output.
R5_sensitivity_figure, R5_sensitivity_axes = plt.subplots(
    1, 3, figsize=(12, 3.4), constrained_layout=True,
    gridspec_kw={"width_ratios": [1.2, 1.0, 1.0]},
)
R5_metric_table = CONSENSUS_METRICS.set_index("k").loc[[6, 8]]
R5_sensitivity_axes[0].bar(np.arange(2)-0.18, R5_metric_table["PAC_0.1_0.9"], 0.36,
                           color="#4477AA", label="PAC (lower)")
R5_sensitivity_axes[0].bar(np.arange(2)+0.18, 1-R5_metric_table["consensus_silhouette"], 0.36,
                           color="#CC6677", label="1 − silhouette (lower)")
R5_sensitivity_axes[0].set_xticks(np.arange(2), ["K=6", "K=8"])
R5_sensitivity_axes[0].set_title("A  Stability diagnostics", loc="left", fontweight="bold")
R5_sensitivity_axes[0].legend(frameon=False, fontsize=7)
for axis, k in zip(R5_sensitivity_axes[1:], [6, 8]):
    counts = np.sort(np.bincount(np.asarray(CONSENSUS_SOLUTIONS[k]["labels"], dtype=int)))[::-1]
    axis.bar(np.arange(len(counts)), counts, color="#4477AA")
    axis.set_xticks(np.arange(len(counts)), [f"FG{i + 1}" for i in range(len(counts))])
    axis.set_ylabel("Latent factors")
    axis.set_title(f"{'B' if k == 6 else 'C'}  K={k} group sizes", loc="left", fontweight="bold")
for suffix in ("png", "pdf", "svg"):
    out = NOTEBOOK_DIR / "figs/individual/Figure_5" / f"Figure_5_supplementary_B.{suffix}"
    out.parent.mkdir(parents=True, exist_ok=True)
    save_kwargs = {"facecolor": "white", "bbox_inches": "tight"}
    if suffix == "png":
        R5_sensitivity_figure.savefig(out, dpi=R5_EXPORT_DPI, **save_kwargs)
    else:
        with matplotlib.rc_context(R5_VECTOR_TEXT_RCPARAMS):
            R5_sensitivity_figure.savefig(out, **save_kwargs)
plt.show(); plt.close(R5_sensitivity_figure)

# Part III — H–W connections across retained epithelial cell types

W supplies biological meaning and H supplies continuous cell-level usage. Every categorical summary uses the five fixed cell-type names.

## 10. Assign the shared factors and expose H activity

The 50 aligned factors are ordered by their K=8 component cluster for display. Factor 12 is the qualified NG5* singleton.

In [ ]:
# K=8 reconstruction, singleton qualification, and cell-type connections
R5_PRIMARY_K = 8
R5_TOP_N = TOP_N
R5_OUTPUT_DPI = 300
R5_CELLTYPE_ORDER = list(CELLTYPE_ORDER)
SUBTYPE_MAPPING = {
    "Epithelial progenitor": "#E69F00",
    "Epithelial luminal": "#009E73",
    "Epithelial basal": "#CC79A7",
    "Epithelial cycling": "#D55E00",
    "IFN-response epithelial": "#56B4E9",
}

R5_COMPONENT_ASSIGNMENTS = COMPONENT_ASSIGNMENTS[["component", "consensus_cluster"]].rename(columns={"consensus_cluster": "program"}).copy()
R5_labels = R5_COMPONENT_ASSIGNMENTS["program"].to_numpy(dtype=int)
R5_component_names = R5_COMPONENT_ASSIGNMENTS["component"].astype(str).to_numpy()
R5_PROGRAM_FACTOR_COUNTS = R5_COMPONENT_ASSIGNMENTS.groupby("program").size().reindex(range(R5_PRIMARY_K))
R5_singleton_rows = R5_COMPONENT_ASSIGNMENTS.loc[
    R5_COMPONENT_ASSIGNMENTS["program"].map(R5_PROGRAM_FACTOR_COUNTS).eq(1)
]
if R5_singleton_rows["component"].tolist() != ["12"]:
    raise RuntimeError("The qualified K=8 singleton must be factor 12")
R5_SINGLETON_FACTOR = "12"
R5_SINGLETON_PROGRAM = int(R5_singleton_rows["program"].iloc[0])
R5_SINGLETON_PROGRAM_ID = f"NG{R5_SINGLETON_PROGRAM + 1}"
R5_PROGRAM_IDS = [f"NG{i + 1}{'*' if i == R5_SINGLETON_PROGRAM else ''}" for i in range(R5_PRIMARY_K)]
R5_COMPONENT_IDS = [f"CC{i + 1}" for i in range(R5_PRIMARY_K)]

# Order the 50 continuous H factors by their component cluster.
R5_FACTOR_TABLE = R5_COMPONENT_ASSIGNMENTS.copy()
R5_FACTOR_TABLE["factor_number"] = R5_FACTOR_TABLE["component"].astype(int)
R5_FACTOR_TABLE = R5_FACTOR_TABLE.sort_values(
    ["program", "factor_number"], kind="mergesort"
).reset_index(drop=True)
R5_H_FACTOR_NUMBERS = R5_FACTOR_TABLE["factor_number"].to_numpy()
R5_H_FACTOR_GROUPS = R5_FACTOR_TABLE["program"].to_numpy(dtype=int)
R5_H_FACTOR_LABELS = [
    f"H factor {factor}{'*' if factor == 12 else ''}"
    for factor in R5_H_FACTOR_NUMBERS
]
R5_H_BY_CELLTYPE_GROUPED = H_CELLTYPE_LOG_Z.reindex(
    [f"H_graph_{factor}" for factor in R5_H_FACTOR_NUMBERS]
).copy()
R5_H_FACTOR_BOUNDARIES = (
    np.flatnonzero(np.diff(R5_H_FACTOR_GROUPS) != 0) + 0.5
)
R5_H_FACTOR_CENTERS = {
    group: float(np.flatnonzero(R5_H_FACTOR_GROUPS == group).mean())
    for group in range(R5_PRIMARY_K)
}

fig_h_grouped, ax_h_grouped = plt.subplots(
    figsize=(8, 9), constrained_layout=True
)
h_values = R5_H_BY_CELLTYPE_GROUPED.to_numpy(dtype=float)
h_limit = max(float(np.nanpercentile(np.abs(h_values), 98)), 1e-6)
h_image = ax_h_grouped.imshow(
    h_values, aspect="auto", interpolation="nearest", cmap="RdBu_r",
    vmin=-h_limit, vmax=h_limit, rasterized=True,
)
ax_h_grouped.set_xticks(range(len(R5_CELLTYPE_ORDER)))
ax_h_grouped.set_xticklabels(R5_CELLTYPE_ORDER, rotation=45, ha="right")
ax_h_grouped.set_yticks(range(50))
ax_h_grouped.set_yticklabels(R5_H_FACTOR_LABELS, fontsize=5.5)
for boundary in R5_H_FACTOR_BOUNDARIES:
    ax_h_grouped.axhline(boundary, color="white", linewidth=0.8)
ax_h_grouped.set_title(
    "Mean H-factor activity by retained epithelial cell type",
    loc="left", fontweight="bold",
)
ax_h_grouped.set_xlabel("Retained epithelial cell type")
ax_h_grouped.set_ylabel("Continuous H factors ordered by component cluster")
fig_h_grouped.colorbar(
    h_image, ax=ax_h_grouped,
    label="Row z-score of log(1 + mean H)", fraction=0.035, pad=0.02,
)
display(fig_h_grouped); plt.close(fig_h_grouped)
ARTIFACTS["H_factor_by_celltype_grouped"] = R5_H_BY_CELLTYPE_GROUPED

## 11. Reconstruct node-group activity by cell type and individual cell

The 20 exponential-decay nodes in each program define a representative W profile, which is multiplied by H for every cell.

In [ ]:
R5_SELECTED_PROGRAM_NODES = MIXED_TOP_NODES.rename(
    columns={"consensus_cluster": "program", "cluster_rank": "program_rank"}
).copy()
R5_PROGRAM_PROFILES = np.vstack([
    W.loc[R5_SELECTED_PROGRAM_NODES.loc[
        R5_SELECTED_PROGRAM_NODES["program"].eq(program), "node_name"
    ]].mean(axis=0).to_numpy()
    for program in range(R5_PRIMARY_K)
])
R5_PROGRAM_ACTIVITY_RAW = H @ R5_PROGRAM_PROFILES.T
R5_PROGRAM_ACTIVITY_LOG = np.log1p(np.maximum(R5_PROGRAM_ACTIVITY_RAW, 0))
R5_PROGRAM_ACTIVITY_Z = (
    R5_PROGRAM_ACTIVITY_LOG - R5_PROGRAM_ACTIVITY_LOG.mean(axis=0, keepdims=True)
) / np.maximum(R5_PROGRAM_ACTIVITY_LOG.std(axis=0, keepdims=True), 1e-12)

R5_CELLTYPE_INDEX = CELLTYPE_LABELS.reset_index(drop=True)
R5_PROGRAM_BY_CELLTYPE_MEAN = pd.DataFrame(
    R5_PROGRAM_ACTIVITY_RAW, columns=R5_PROGRAM_IDS
).groupby(R5_CELLTYPE_INDEX, observed=True).mean().reindex(R5_CELLTYPE_ORDER).T
R5_PROGRAM_BY_CELLTYPE_Z = pd.DataFrame(
    row_log_z(R5_PROGRAM_BY_CELLTYPE_MEAN),
    index=R5_PROGRAM_IDS, columns=R5_CELLTYPE_ORDER,
)

display(R5_PROGRAM_BY_CELLTYPE_Z)
inline_heatmap(
    R5_PROGRAM_BY_CELLTYPE_Z,
    "Reconstructed K=8 node-group activity by retained epithelial cell type",
    "Row z-score of log(1 + mean activity)",
)
ARTIFACTS["program_by_celltype_mean"] = R5_PROGRAM_BY_CELLTYPE_MEAN
ARTIFACTS["program_by_celltype_log_z"] = R5_PROGRAM_BY_CELLTYPE_Z

## 12. Shared W/H factor-group activity across cell types

Each CC1–CC8 group averages the H columns assigned by W consensus clustering, making H explicit in the second connection transition.

In [ ]:
# Mean H usage for each component cluster, summarized by cell type.
R5_COMPONENT_ACTIVITY_RAW = np.column_stack([
    H[:, np.flatnonzero(R5_labels == program)].mean(axis=1)
    for program in range(R5_PRIMARY_K)
])
R5_COMPONENT_BY_CELLTYPE_MEAN = pd.DataFrame(
    R5_COMPONENT_ACTIVITY_RAW, columns=R5_COMPONENT_IDS
).groupby(R5_CELLTYPE_INDEX, observed=True).mean().reindex(R5_CELLTYPE_ORDER).T
R5_COMPONENT_BY_CELLTYPE_Z = pd.DataFrame(
    row_log_z(R5_COMPONENT_BY_CELLTYPE_MEAN),
    index=R5_COMPONENT_IDS, columns=R5_CELLTYPE_ORDER,
)

R5_W_HEATMAP = W_BY_COMPONENT_LOG_ZSCORE.loc[MIXED_TOP_NODES["node_name"].astype(str)].copy()
R5_W_ROW_PROGRAMS = MIXED_TOP_NODES["consensus_cluster"].astype(int).to_numpy()
R5_W_PROGRAM_BOUNDARIES = np.flatnonzero(np.diff(R5_W_ROW_PROGRAMS) != 0) + 0.5
R5_W_PROGRAM_CENTERS = {
    program: float(np.flatnonzero(R5_W_ROW_PROGRAMS == program).mean())
    for program in range(R5_PRIMARY_K)
}
R5_PROGRAM_TO_COMPONENT = pd.DataFrame(
    np.vstack([
        R5_W_HEATMAP.iloc[np.flatnonzero(R5_W_ROW_PROGRAMS == program)].mean(axis=0).to_numpy()
        for program in range(R5_PRIMARY_K)
    ]), index=R5_PROGRAM_IDS, columns=R5_COMPONENT_IDS,
)

display(R5_COMPONENT_BY_CELLTYPE_Z)
inline_heatmap(
    R5_COMPONENT_BY_CELLTYPE_Z,
    "Shared W/H factor-group activity by retained epithelial cell type",
    "Row z-score of log(1 + mean H)",
)
ARTIFACTS["shared_factor_group_by_celltype_mean"] = R5_COMPONENT_BY_CELLTYPE_MEAN
ARTIFACTS["shared_factor_group_by_celltype_log_z"] = R5_COMPONENT_BY_CELLTYPE_Z

## 13. Deterministic all-cell ordering

Cells are ordered by fixed cell type and then descriptive sample metadata; the metadata do not define analytical groups.

In [ ]:
# One deterministic cell order: cell type first, then descriptive metadata.
R5_METADATA_COLUMNS = ["Site", "metastatic_status", "Patient IDs", "Treatment", "library", "celltype_confidence", "cell_join_key"]
R5_MISSING_METADATA_COLUMNS = [column for column in R5_METADATA_COLUMNS if column not in cell_metadata.columns]
if R5_MISSING_METADATA_COLUMNS:
    raise ValueError(f"Missing Figure 5D metadata columns: {R5_MISSING_METADATA_COLUMNS}")
R5_CELLTYPE_RANK = CELLTYPE_LABELS.map({name: i for i, name in enumerate(R5_CELLTYPE_ORDER)}).to_numpy()
R5_CELL_ORDER = np.lexsort((
    cell_metadata["cell_join_key"].astype(object).where(cell_metadata["cell_join_key"].notna(),"Unknown").astype(str).to_numpy(),
    cell_metadata["library"].astype(object).where(cell_metadata["library"].notna(),"Unknown").astype(str).to_numpy(),
    cell_metadata["Treatment"].astype(object).where(cell_metadata["Treatment"].notna(),"Unknown").astype(str).to_numpy(),
    cell_metadata["Patient IDs"].astype(object).where(cell_metadata["Patient IDs"].notna(),"Unknown").astype(str).to_numpy(),
    cell_metadata["metastatic_status"].astype(object).where(cell_metadata["metastatic_status"].notna(),"Unknown").astype(str).to_numpy(),
    cell_metadata["Site"].astype(object).where(cell_metadata["Site"].notna(),"Unknown").astype(str).to_numpy(),
    R5_CELLTYPE_RANK,
))
R5_ORDERED_METADATA = cell_metadata.iloc[R5_CELL_ORDER].reset_index(drop=True).copy()
R5_ORDERED_CELLTYPES = CELLTYPE_LABELS.iloc[R5_CELL_ORDER].to_numpy()
R5_CELLTYPE_BOUNDARIES = np.flatnonzero(R5_ORDERED_CELLTYPES[1:] != R5_ORDERED_CELLTYPES[:-1]) + 1

## 14. node group → component cluster → retained cell type

All 64 program-to-group and 40 group-to-cell-type associations are retained as signed associations rather than mass-conserving flows.

In [ ]:
R5_CONNECTION_RECORDS = []
for program_id in R5_PROGRAM_IDS:
    for group_id in R5_COMPONENT_IDS:
        value = float(R5_PROGRAM_TO_COMPONENT.loc[program_id, group_id])
        R5_CONNECTION_RECORDS.append({
            "source_layer": "W_program",
            "target_layer": "shared_W_H_factor_group",
            "source_id": program_id,
            "target_id": group_id,
            "raw_value": value,
            "absolute_strength": abs(value),
        })
for group_id in R5_COMPONENT_IDS:
    for cell_type in R5_CELLTYPE_ORDER:
        value = float(R5_COMPONENT_BY_CELLTYPE_Z.loc[group_id, cell_type])
        R5_CONNECTION_RECORDS.append({
            "source_layer": "shared_W_H_factor_group",
            "target_layer": "retained_epithelial_cell_type",
            "source_id": group_id,
            "target_id": cell_type,
            "raw_value": value,
            "absolute_strength": abs(value),
        })
R5_CONNECTIONS = pd.DataFrame(R5_CONNECTION_RECORDS)
show_table(R5_CONNECTIONS, "All retained annotation-node-filtered H–W associations", rows=30)
ARTIFACTS["celltype_H_W_connections"] = R5_CONNECTIONS

## 15. Static all-edge signed association network

In [ ]:
from matplotlib.path import Path as R5_Path
from matplotlib.patches import PathPatch as R5_PathPatch

def R5_analysis_curve(axis, x0, y0, x1, y1, value, layer_max):
    strength = abs(value) / max(layer_max, 1e-12)
    midpoint = (x0 + x1) / 2
    path = R5_Path(
        [(x0, y0), (midpoint, y0), (midpoint, y1), (x1, y1)],
        [R5_Path.MOVETO, R5_Path.CURVE4, R5_Path.CURVE4, R5_Path.CURVE4],
    )
    axis.add_patch(R5_PathPatch(
        path, facecolor="none",
        edgecolor="#B54A4A" if value >= 0 else "#4477AA",
        linewidth=0.2 + 2.6 * strength,
        alpha=0.05 + 0.58 * strength,
        linestyle="-" if value >= 0 else "--",
    ))

fig_connections, ax_connections = plt.subplots(figsize=(14, 7))
y_program = np.linspace(0.93, 0.07, 8)
y_group = np.linspace(0.93, 0.07, 8)
y_celltype = np.linspace(0.88, 0.12, 5)
n2c = R5_PROGRAM_TO_COMPONENT.to_numpy(dtype=float)
c2t = R5_COMPONENT_BY_CELLTYPE_Z.to_numpy(dtype=float)
for i in range(8):
    for j in range(8):
        R5_analysis_curve(
            ax_connections, 0.10, y_program[i], 0.47, y_group[j],
            n2c[i, j], np.abs(n2c).max(),
        )
for i in range(8):
    for j in range(5):
        R5_analysis_curve(
            ax_connections, 0.53, y_group[i], 0.90, y_celltype[j],
            c2t[i, j], np.abs(c2t).max(),
        )
for i, label in enumerate(R5_PROGRAM_IDS):
    ax_connections.scatter(0.08, y_program[i], s=65, color=CLUSTER_COLORS[i], zorder=3)
    ax_connections.text(0.055, y_program[i], label, ha="right", va="center")
for i, label in enumerate(R5_COMPONENT_IDS):
    ax_connections.scatter(0.50, y_group[i], s=70, color="#D07A3D", zorder=3)
    ax_connections.text(0.50, y_group[i], label, ha="center", va="center", color="white")
for i, label in enumerate(R5_CELLTYPE_ORDER):
    ax_connections.scatter(0.92, y_celltype[i], s=75, color=SUBTYPE_MAPPING[label], zorder=3)
    ax_connections.text(0.94, y_celltype[i], label, ha="left", va="center")
ax_connections.text(0.08, 1.01, "Node groups", ha="center", fontweight="bold")
ax_connections.text(0.50, 1.01, "Shared W/H component clusters", ha="center", fontweight="bold")
ax_connections.text(0.92, 1.01, "Retained epithelial cell types", ha="center", fontweight="bold")
ax_connections.set_xlim(0, 1.18); ax_connections.set_ylim(0, 1.05)
ax_connections.axis("off")
ax_connections.set_title(
    "All-edge signed H–W association network", loc="left", fontweight="bold"
)
display(fig_connections); plt.close(fig_connections)
ARTIFACTS["celltype_H_W_network"] = fig_connections

## 16. Literature-prioritized metabolic graph-node scores

Ten biologically motivated, nonredundant graph nodes are reconstructed. Model support is reported transparently, cell-level distributions are descriptive, and patient coverage is explicit. No inferential tests are applied because cells are not independent biological replicates.

In [ ]:
# Literature relevance is the primary, predeclared selection criterion.
# Cell-type range and decay membership are supporting model evidence only.
R5_FEATURE_DEFINITIONS = [
    ("GO_term", "folate transmembrane transport (GO:0098838)",
     "Folate transmembrane transport", "Epithelial progenitor",
     "Folate transport supporting biosynthetic demand", "GO:0098838",
     "Weak descriptive progenitor association; model-inferred activity"),
    ("GO_term", "hydrogen sulfide metabolic process (GO:0070813)",
     "Hydrogen sulfide metabolism", "Epithelial luminal",
     "Sulfur-amino-acid and hydrogen-sulfide metabolism", "GO:0070813",
     "Model-inferred process activity; not measured H2S abundance"),
    ("GO_term", "basement membrane assembly (GO:0070831)",
     "Basement membrane assembly", "Epithelial basal",
     "Basal epithelial–matrix organization", "GO:0070831",
     "P3-dominated descriptive association"),
    ("GO_term", "DNA replication, synthesis of RNA primer (GO:0006269)",
     "DNA replication: RNA-primer synthesis", "Epithelial cycling",
     "DNA replication and proliferative activity", "GO:0006269",
     "P5-dominated descriptive association"),
    ("geneset", "HALLMARK_INTERFERON_GAMMA_RESPONSE",
     "Hallmark interferon-γ response", "IFN-response epithelial",
     "Interferon-response transcriptional program", "MSigDB hallmark",
     "P4-dominated descriptive association"),
]

R5_FEATURE_RATIONALE = pd.DataFrame(
    R5_FEATURE_DEFINITIONS,
    columns=["node_kind", "node_name", "display_label", "target_cell_type",
             "biological_theme", "evidence_id", "interpretation_limit"],
)
R5_FEATURE_RATIONALE["selection_basis"] = "predeclared functional non-gene representative"
R5_ALL_FEATURES = [
    tuple(row) for row in R5_FEATURE_RATIONALE[["node_kind", "node_name"]].to_numpy()
]
R5_ALL_FEATURE_NAMES = [name for _, name in R5_ALL_FEATURES]
R5_MISSING_ALL_FEATURES = sorted(set(R5_ALL_FEATURE_NAMES) - set(W.index))
if R5_MISSING_ALL_FEATURES:
    raise ValueError(f"Selected non-gene nodes absent from W: {R5_MISSING_ALL_FEATURES}")
R5_ALL_FEATURE_RAW = H @ W.loc[R5_ALL_FEATURE_NAMES].to_numpy(dtype=float).T
R5_ALL_FEATURE_LOG = np.log1p(np.maximum(R5_ALL_FEATURE_RAW, 0))
R5_ALL_FEATURE_Z = (
    R5_ALL_FEATURE_LOG - R5_ALL_FEATURE_LOG.mean(axis=0, keepdims=True)
) / np.maximum(R5_ALL_FEATURE_LOG.std(axis=0, keepdims=True), 1e-12)

R5_H_CELLTYPE_MEAN = np.vstack([
    H[R5_CELLTYPE_INDEX.eq(cell_type).to_numpy()].mean(axis=0)
    for cell_type in R5_CELLTYPE_ORDER
])
R5_ALL_NODE_CELLTYPE_LOG = np.log1p(W.to_numpy(dtype=float) @ R5_H_CELLTYPE_MEAN.T)
R5_NODE_CELLTYPE_SUPPORT = node_metadata[["node_name", "node_kind"]].copy()
R5_NODE_CELLTYPE_SUPPORT["celltype_log_range"] = np.ptp(R5_ALL_NODE_CELLTYPE_LOG, axis=1)
R5_NODE_CELLTYPE_SUPPORT["celltype_category_rank"] = (
    R5_NODE_CELLTYPE_SUPPORT.groupby("node_kind")["celltype_log_range"]
    .rank(method="min", ascending=False).astype(int)
)
R5_NODE_CELLTYPE_SUPPORT["category_size"] = (
    R5_NODE_CELLTYPE_SUPPORT.groupby("node_kind")["node_name"].transform("size")
)
R5_NODE_CELLTYPE_SUPPORT["celltype_category_percentile"] = 1 - (
    (R5_NODE_CELLTYPE_SUPPORT["celltype_category_rank"] - 1)
    / R5_NODE_CELLTYPE_SUPPORT["category_size"].clip(lower=1)
)
R5_FEATURE_RATIONALE = R5_FEATURE_RATIONALE.merge(
    R5_NODE_CELLTYPE_SUPPORT, on=["node_name", "node_kind"],
    how="left", validate="one_to_one",
)
R5_FEATURE_RATIONALE["decay_selected"] = R5_FEATURE_RATIONALE["node_name"].isin(
    set(MIXED_TOP_NODES["node_name"].astype(str))
)
R5_TOP10_CORRELATION = pd.DataFrame(
    np.corrcoef(R5_ALL_FEATURE_LOG, rowvar=False),
    index=R5_ALL_FEATURE_NAMES, columns=R5_ALL_FEATURE_NAMES,
)
R5_TOP10_MAX_ABS_CORRELATION = float(np.abs(
    R5_TOP10_CORRELATION.to_numpy()[np.triu_indices(len(R5_ALL_FEATURE_NAMES), k=1)]
).max())

R5_FEATURE_LONG = pd.DataFrame(R5_ALL_FEATURE_Z, columns=R5_ALL_FEATURE_NAMES)
R5_FEATURE_LONG["cell_type"] = R5_CELLTYPE_INDEX.to_numpy()
R5_FEATURE_LONG["patient"] = cell_metadata["Patient IDs"].astype(str).to_numpy()
R5_FEATURE_MELTED = R5_FEATURE_LONG.melt(
    id_vars=["cell_type", "patient"], var_name="node_name", value_name="activity_z"
)
R5_FEATURE_CELLTYPE_SUMMARY = (
    R5_FEATURE_MELTED.groupby(["node_name", "cell_type"], observed=True)
    .agg(cells=("activity_z", "size"), patients=("patient", "nunique"),
         mean_z=("activity_z", "mean"), median_z=("activity_z", "median"),
         q25_z=("activity_z", lambda x: x.quantile(0.25)),
         q75_z=("activity_z", lambda x: x.quantile(0.75)))
    .reset_index()
)
R5_PATIENT_CELLTYPE_COVERAGE = pd.crosstab(
    cell_metadata["Patient IDs"].astype(str), R5_CELLTYPE_INDEX
).reindex(columns=R5_CELLTYPE_ORDER, fill_value=0)
show_table(R5_FEATURE_RATIONALE, "Functional non-gene representative-node rationale", rows=10)
show_table(R5_FEATURE_CELLTYPE_SUMMARY, "Descriptive activity summaries", rows=50)
display(R5_PATIENT_CELLTYPE_COVERAGE)
print(
    "Inference is intentionally omitted: patient-by-cell-type coverage is sparse, "
    "and cell-level observations are descriptive rather than biological replicates."
)

R5_SELECTED_FEATURES = [
    (row.node_kind, row.node_name, row.display_label)
    for row in R5_FEATURE_RATIONALE.itertuples(index=False)
]

R5_FEATURE_NAMES = [name for _, name, _ in R5_SELECTED_FEATURES]
R5_FEATURE_POSITIONS = [R5_ALL_FEATURE_NAMES.index(name) for name in R5_FEATURE_NAMES]
R5_FEATURE_Z = R5_ALL_FEATURE_Z[:, R5_FEATURE_POSITIONS]

fig_non_gene, axes_non_gene = plt.subplots(1, 5, figsize=(17, 3.8), constrained_layout=True)
for feature_index, axis in enumerate(axes_non_gene.flat):
    groups = [
        R5_ALL_FEATURE_Z[R5_CELLTYPE_INDEX.eq(cell_type).to_numpy(), feature_index]
        for cell_type in R5_CELLTYPE_ORDER
    ]
    parts = axis.violinplot(groups, positions=range(5), showextrema=False, widths=0.85)
    for body, cell_type in zip(parts["bodies"], R5_CELLTYPE_ORDER):
        body.set_facecolor(SUBTYPE_MAPPING[cell_type])
        body.set_alpha(0.65); body.set_edgecolor("none")
    axis.boxplot(
        groups, positions=range(5), widths=0.16, showfliers=False,
        patch_artist=True, boxprops=dict(facecolor="white", linewidth=0.6),
        medianprops=dict(color="black", linewidth=0.8),
    )
    axis.set_title(R5_FEATURE_RATIONALE.iloc[feature_index]["display_label"], fontsize=7.5)
    axis.set_xticks(range(5))
    axis.set_xticklabels(R5_CELLTYPE_ORDER, rotation=45, ha="right", fontsize=5.5)
    if feature_index % 5 == 0:
        axis.set_ylabel("scKNIFE-inferred graph-node score (z-score)", fontsize=7)
    axis.grid(False)
    axis.spines["top"].set_visible(False); axis.spines["right"].set_visible(False)
fig_non_gene.suptitle(
    "Functional non-gene graph-node scores across epithelial cell types",
    fontsize=11, fontweight="bold",
)
R5_TOP10_OUTPUT_DIR = NOTEBOOK_DIR / "figs/individual/Figure_5"
R5_DERIVED_OUTPUT_DIR = NOTEBOOK_DIR / "data/Figure_5/derived"
R5_TOP10_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
R5_DERIVED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
R5_TOP10_BASE = R5_TOP10_OUTPUT_DIR / "Figure_5_supplementary_C"
fig_non_gene.savefig(
    R5_TOP10_BASE.with_suffix(".png"),
    dpi=R5_EXPORT_DPI,
    bbox_inches="tight",
    facecolor="white",
)
with matplotlib.rc_context(R5_VECTOR_TEXT_RCPARAMS):
    fig_non_gene.savefig(
        R5_TOP10_BASE.with_suffix(".pdf"), bbox_inches="tight", facecolor="white"
    )
    fig_non_gene.savefig(
        R5_TOP10_BASE.with_suffix(".svg"), bbox_inches="tight", facecolor="white"
    )
R5_FEATURE_RATIONALE.to_csv(
    R5_DERIVED_OUTPUT_DIR / "Figure_5_top10_literature_rationale.csv", index=False
)
R5_TOP10_CORRELATION.to_csv(
    R5_DERIVED_OUTPUT_DIR / "Figure_5_top10_reconstructed_score_correlations.csv"
)
display(fig_non_gene); plt.close(fig_non_gene)
ARTIFACTS["non_gene_activity_z"] = R5_ALL_FEATURE_Z
ARTIFACTS["non_gene_celltype_summary"] = R5_FEATURE_CELLTYPE_SUMMARY
ARTIFACTS["top10_literature_rationale"] = R5_FEATURE_RATIONALE
ARTIFACTS["top10_correlations"] = R5_TOP10_CORRELATION
ARTIFACTS["patient_celltype_coverage"] = R5_PATIENT_CELLTYPE_COVERAGE

## 17. Final annotation-node-filtered audit

In [ ]:
R5_FINAL_AUDIT = pd.Series({
    "W_nodes_39247": W.shape == EXPECTED_W_SHAPE,
    "excluded_annotation_node_counts": EXCLUDED_NODE_KIND_COUNTS.to_dict() == EXPECTED_EXCLUDED_NODE_KIND_COUNTS,
    "eligible_W_nodes_39035": int(ELIGIBLE_MASK.sum()) == EXPECTED_ELIGIBLE_W_NODES,
    "eligible_node_kinds_exclude_annotations": not NODE_METADATA_ELIGIBLE["node_kind"].astype(str).isin(EXCLUDED_W_NODE_KINDS).any(),
    "selected_node_kinds_exclude_annotations": not MIXED_TOP_NODES["node_kind"].astype(str).isin(EXCLUDED_W_NODE_KINDS).any(),
    "H_cells_43574_factors_50": H.shape == EXPECTED_H_SHAPE,
    "five_fixed_celltypes": set(CELLTYPE_LABELS) == set(R5_CELLTYPE_ORDER),
    "K8_program_alignment_complete": set(PROGRAM_ALIGNMENT["program"]) == set(range(8)),
    "K8_program_alignment_jaccard_at_least_0p5": PROGRAM_ALIGNMENT["factor_membership_jaccard"].ge(0.5).all(),
    "K8_selected": SELECTED_K == R5_PRIMARY_K,
    "all_50_H_factors_assigned_once": (
        len(R5_FACTOR_TABLE) == 50 and R5_FACTOR_TABLE["component"].is_unique
    ),
    "H_factor_celltype_matrix_50x5": R5_H_BY_CELLTYPE_GROUPED.shape == (50, 5),
    "factor12_is_unique_singleton": (
        R5_SINGLETON_FACTOR == "12"
        and len(R5_singleton_rows) == 1
        and int(R5_PROGRAM_FACTOR_COUNTS.loc[R5_SINGLETON_PROGRAM]) == 1
    ),
    "decay_nodes_160": len(MIXED_TOP_NODES) == 160,
    "twenty_nodes_per_program": MIXED_TOP_NODES.groupby(
        "consensus_cluster"
    ).size().eq(20).all(),
    "decay_nodes_globally_unique": MIXED_TOP_NODES["node_name"].is_unique,
    "program_celltype_matrix_8x5": R5_PROGRAM_BY_CELLTYPE_Z.shape == (8, 5),
    "shared_factor_celltype_matrix_8x5": R5_COMPONENT_BY_CELLTYPE_Z.shape == (8, 5),
    "program_activity_43574x8": R5_PROGRAM_ACTIVITY_Z.shape == (43574, 8),
    "celltype_boundaries_4": len(R5_CELLTYPE_BOUNDARIES) == 4,
    "celltype_connections_104": len(R5_CONNECTIONS) == 8 * 8 + 8 * 5,
    "five_functional_non_gene_features": R5_ALL_FEATURE_Z.shape == (43574, 5),
    "five_features_unique": len(set(R5_ALL_FEATURE_NAMES)) == 5,
    "functional_non_gene_kinds": set(R5_FEATURE_RATIONALE["node_kind"]).issubset({"GO_term", "pathway", "task", "metabolite", "reaction", "geneset"}),
    "representatives_nonredundant_below_0p98": R5_TOP10_MAX_ABS_CORRELATION < 0.98,
    "representative_decay_support_recomputed": R5_FEATURE_RATIONALE["decay_selected"].equals(
        R5_FEATURE_RATIONALE["node_name"].isin(set(MIXED_TOP_NODES["node_name"].astype(str)))
    ),
    "five_main_panel_features": R5_FEATURE_Z.shape == (43574, 5),
    "patient_coverage_7x5": R5_PATIENT_CELLTYPE_COVERAGE.shape == (7, 5),
    "sparse_coverage_documented": (R5_PATIENT_CELLTYPE_COVERAGE == 0).any().any(),
    "all_values_finite": all(np.isfinite(values).all() for values in [
        R5_H_BY_CELLTYPE_GROUPED.to_numpy(), R5_PROGRAM_ACTIVITY_Z,
        R5_PROGRAM_BY_CELLTYPE_Z.to_numpy(),
        R5_COMPONENT_BY_CELLTYPE_Z.to_numpy(), R5_ALL_FEATURE_Z,
        R5_CONNECTIONS["raw_value"].to_numpy(),
    ]),
}, name="passed")
display(R5_FINAL_AUDIT.to_frame())
if not R5_FINAL_AUDIT.all():
    raise RuntimeError("Annotation-node-filtered H→W→H–W audit failed")
ARTIFACTS["final_celltype_H_W_audit"] = R5_FINAL_AUDIT
progress("Annotation-node-filtered H→W→H–W workflow validated")

## 10. Standalone node-group dictionary for sectionwise review

This review export keeps the validated K=8 assignments and exponential-decay node selection unchanged. The main dictionary shows the five leading nodes from each 20-node program set; the complete 160-node heatmap is exported separately for supplementary review. The existing authoritative panel B and A–G composite are not overwritten.

In [ ]:
from extractor_scripts.render_figure5_w_program_review import render_w_program_review

R5W_REVIEW = render_w_program_review(
    notebook_dir=NOTEBOOK_DIR,
    mixed_top_nodes=MIXED_TOP_NODES,
    program_factor_counts=R5_PROGRAM_FACTOR_COUNTS,
    w_by_component_log_zscore=W_BY_COMPONENT_LOG_ZSCORE,
    component_ids=R5_COMPONENT_IDS,
    category_colors=CATEGORY_COLORS,
    selected_k=R5_PRIMARY_K,
    singleton_program=R5_SINGLETON_PROGRAM,
    singleton_factor=R5_SINGLETON_FACTOR,
    decay_tau=DECAY_TAU,
    decay_cumulative_mass=DECAY_CUMULATIVE_MASS,
    max_nodes_per_cluster_kind=MAX_NODES_PER_CLUSTER_KIND,
    top_n=TOP_N,
)
display(R5W_REVIEW["checks"].to_frame())
display(R5W_REVIEW["visible_nodes"][[
    "program_id", "program_name", "factor_count",
    "cluster_rank", "node_kind", "node_name",
    "enrichment_delta_z", "decay_weight", "selection_reason",
]])
ARTIFACTS["W_program_review_full_nodes"] = R5W_REVIEW["full_nodes"]
ARTIFACTS["W_program_review_visible_nodes"] = R5W_REVIEW["visible_nodes"]
ARTIFACTS["W_program_review_checks"] = R5W_REVIEW["checks"]
progress(f"Standalone node-group review exports complete: {R5W_REVIEW['manifest_path']}")


# Final Figure 5 — continuous H factors, K=8 node groups, and retained epithelial cell types

Panels A–H follow H factors through W definitions and the validated node-group dictionary to component clusters, epithelial cell-type activity, and functional non-gene nodes.


In [ ]:
R5F_PATIENT_CELLTYPE_COUNTS = pd.crosstab(
    cell_metadata["Patient IDs"].astype(str), R5_CELLTYPE_INDEX
).reindex(columns=R5_CELLTYPE_ORDER, fill_value=0)
R5F_PATIENT_PROGRAM_MEANS = pd.DataFrame(R5_PROGRAM_ACTIVITY_RAW, columns=R5_PROGRAM_IDS)
R5F_PATIENT_PROGRAM_MEANS["patient"] = cell_metadata["Patient IDs"].astype(str).to_numpy()
R5F_PATIENT_PROGRAM_MEANS["cell_type"] = R5_CELLTYPE_INDEX.to_numpy()
R5F_PATIENT_PROGRAM_MEANS = R5F_PATIENT_PROGRAM_MEANS.groupby(["patient", "cell_type"], observed=True)[R5_PROGRAM_IDS].mean().reset_index()
R5F_EQUAL_PATIENT_PROGRAM_MEAN = R5F_PATIENT_PROGRAM_MEANS.groupby("cell_type", observed=True)[R5_PROGRAM_IDS].mean().reindex(R5_CELLTYPE_ORDER).T
R5F_EQUAL_PATIENT_PROGRAM_Z = pd.DataFrame(row_log_z(R5F_EQUAL_PATIENT_PROGRAM_MEAN), index=R5_PROGRAM_IDS, columns=R5_CELLTYPE_ORDER)
R5F_PATIENT_SENSITIVITY = (R5_PROGRAM_BY_CELLTYPE_Z - R5F_EQUAL_PATIENT_PROGRAM_Z).abs()

R5F_PATIENT_CELLTYPE_COUNTS.to_csv(R5_DERIVED_OUTPUT_DIR / "Figure_5_patient_celltype_census.csv")
R5F_EQUAL_PATIENT_PROGRAM_Z.to_csv(R5_DERIVED_OUTPUT_DIR / "Figure_5_equal_patient_NG_activity_z.csv")
R5F_PATIENT_SENSITIVITY.to_csv(R5_DERIVED_OUTPUT_DIR / "Figure_5_cell_vs_equal_patient_sensitivity.csv")
R5_FEATURE_RATIONALE.to_csv(R5_DERIVED_OUTPUT_DIR / "Figure_5_representative_non_gene_nodes.csv", index=False)


def R5F_draw_supplementary_D(axis):
    figure, spec = axis.figure, axis.get_subplotspec(); axis.remove()
    sub = spec.subgridspec(1, 3, width_ratios=[1.25, 1, 1], wspace=.52)
    census_ax = figure.add_subplot(sub[0, 0])
    values = np.log10(R5F_PATIENT_CELLTYPE_COUNTS.to_numpy() + 1)
    census_image = census_ax.imshow(values, aspect="auto", cmap="viridis", rasterized=True)
    census_ax.set_title("D  Patient representation", loc="left", fontsize=R5F_PANEL_PT, fontweight="bold")
    census_ax.set_xticks(range(5), [v.replace("Epithelial ", "") for v in R5_CELLTYPE_ORDER], rotation=45, ha="right")
    census_ax.set_yticks(range(len(R5F_PATIENT_CELLTYPE_COUNTS)), [R5F_PATIENT_DISPLAY[value] for value in R5F_PATIENT_CELLTYPE_COUNTS.index], fontsize=R5F_SMALL_PT)
    figure.colorbar(census_image, ax=census_ax, fraction=.04, pad=.02, label="log10(cells + 1)")
    cell_ax = figure.add_subplot(sub[0, 1])
    equal_ax = figure.add_subplot(sub[0, 2])
    limit = max(R5F_limit(R5_PROGRAM_BY_CELLTYPE_Z.to_numpy(), 99), R5F_limit(R5F_EQUAL_PATIENT_PROGRAM_Z.to_numpy(), 99))
    for target, matrix, title in [(cell_ax, R5_PROGRAM_BY_CELLTYPE_Z, "Cell-weighted"), (equal_ax, R5F_EQUAL_PATIENT_PROGRAM_Z, "Equal-patient")]:
        target.imshow(matrix.to_numpy(), aspect="auto", cmap=R5F_CMAP, vmin=-limit, vmax=limit, rasterized=True)
        target.set_title(title, fontsize=R5F_AXIS_PT)
        target.set_xticks(range(5), [v.replace("Epithelial ", "") for v in R5_CELLTYPE_ORDER], rotation=45, ha="right")
        target.set_yticks(range(8), R5_PROGRAM_IDS if target is cell_ax else [])
        target.set_xlabel("Epithelial cell type")


# R5F — Annotation-node-filtered A–H publication figure and exports
from matplotlib.patches import Rectangle as R5F_Rectangle
from matplotlib.path import Path as R5F_Path
from matplotlib.patches import PathPatch as R5F_PathPatch
from extractor_scripts.render_figure5_w_program_review import (
    draw_w_program_dictionary as R5F_draw_program_dictionary,
)

# ─── Figure parameters ───
R5F_WIDTH_MM = 180
R5F_HEIGHT_MM = 620
R5F_SUPPLEMENT_HEIGHT_MM = 228
R5F_PANEL_PT = 10
R5F_AXIS_PT = 8
R5F_TICK_PT = 6
R5F_SMALL_PT = 5.2
R5F_FACTOR_PT = 4.2
R5F_DPI = R5_EXPORT_DPI
R5F_FONT_FAMILY = R5_FONT_FAMILY
R5F_PDF_FONT_TYPE = R5_PDF_FONT_TYPE
R5F_SVG_FONT_TYPE = R5_SVG_FONT_TYPE
R5F_VECTOR_TEXT_RCPARAMS = {
    "font.family": R5F_FONT_FAMILY,
    "pdf.fonttype": R5F_PDF_FONT_TYPE,
    "svg.fonttype": R5F_SVG_FONT_TYPE,
}
R5F_CMAP = "RdBu_r"
R5F_REVISION = "figure5_supplementary_ef_side_by_side_20260828"
R5F_PATIENT_ID_MAPPING_PATH = NOTEBOOK_DIR / "data/Figure_4/patient_id_mapping.csv"
R5F_PATIENT_ID_TABLE = pd.read_csv(R5F_PATIENT_ID_MAPPING_PATH)
if list(R5F_PATIENT_ID_TABLE.columns) != ["display_id", "patient_id"]:
    raise RuntimeError("Unexpected patient-ID mapping schema")
if (not R5F_PATIENT_ID_TABLE["display_id"].is_unique or not R5F_PATIENT_ID_TABLE["patient_id"].is_unique):
    raise RuntimeError("Patient-ID mapping must be one-to-one")
R5F_PATIENT_ID_TABLE["display_number"] = R5F_PATIENT_ID_TABLE["display_id"].str.extract(r"(\d+)").astype(int)
R5F_PATIENT_ID_TABLE = R5F_PATIENT_ID_TABLE.sort_values("display_number").reset_index(drop=True)
R5F_PATIENT_DISPLAY = dict(zip(R5F_PATIENT_ID_TABLE["patient_id"], R5F_PATIENT_ID_TABLE["display_id"]))
R5F_DISPLAY_TO_PATIENT = dict(zip(R5F_PATIENT_ID_TABLE["display_id"], R5F_PATIENT_ID_TABLE["patient_id"]))
R5F_PATIENT_ORDER = R5F_PATIENT_ID_TABLE["patient_id"].tolist()
R5F_PATIENT_DISPLAY_ORDER = R5F_PATIENT_ID_TABLE["display_id"].tolist()
R5F_OBSERVED_PATIENTS = set(cell_metadata["Patient IDs"].astype(str))
if R5F_OBSERVED_PATIENTS != set(R5F_PATIENT_ORDER):
    raise RuntimeError(f"Patient-ID mapping mismatch: observed={sorted(R5F_OBSERVED_PATIENTS)} mapped={sorted(R5F_PATIENT_ORDER)}")
R5F_ORDERED_PATIENT_CODES = R5_ORDERED_METADATA["Patient IDs"].astype(str).map(R5F_PATIENT_DISPLAY)
if R5F_ORDERED_PATIENT_CODES.isna().any():
    raise RuntimeError("Unmapped patient IDs in Figure 5 metadata")
R5F_PATIENT_CELLTYPE_COUNTS = R5F_PATIENT_CELLTYPE_COUNTS.reindex(R5F_PATIENT_ORDER)
R5F_RENDERER = "Matplotlib fallback (Python 3.9; cnsplots unavailable)"
R5F_MAIN_HEIGHT_RATIOS = (8.20, 2.20, 1.00, 2.20, 1.10, 1.15, 1.55)
R5F_MAIN_HSPACE = 0.50
R5F_MAIN_WSPACE = 0.42
R5F_MAIN_LEFT = 0.085
R5F_MAIN_RIGHT = 0.955
R5F_MAIN_BOTTOM = 0.040
R5F_MAIN_TOP = 0.945
R5F_SITE_ORDER = ("Bx", "Surgery", "Recurrence")
R5F_SITE_COLORS = {"Bx": "#377EB8", "Surgery": "#E41A1C", "Recurrence": "#984EA3"}
R5F_VIOLIN_WIDTH = 0.82
R5F_BOX_WIDTH = 0.14
R5F_MEAN_MARKER_SIZE = 18
R5F_MAIN_NODE_WSPACE = 0.28
# ──────────────────────────

_compiled_dir = NOTEBOOK_DIR / "figs/panel/Figure_5"
_individual_dir = NOTEBOOK_DIR / "figs/individual/Figure_5"
_derived_dir = NOTEBOOK_DIR / "data/Figure_5/derived"
_compiled_dir.mkdir(parents=True, exist_ok=True)
_individual_dir.mkdir(parents=True, exist_ok=True)
_derived_dir.mkdir(parents=True, exist_ok=True)


def R5F_limit(values, percentile=98):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    return (
        max(float(np.percentile(np.abs(finite), percentile)), 0.25)
        if finite.size else 1.0
    )


def R5F_clean(axis):
    axis.grid(False)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.tick_params(labelsize=R5F_TICK_PT)


def R5F_heatmap(
    axis, values, xlabels, ylabels, title, cbar_label, percentile=98
):
    values = np.asarray(values, dtype=float)
    limit = R5F_limit(values, percentile)
    image = axis.imshow(
        values,
        aspect="auto",
        interpolation="nearest",
        cmap=R5F_CMAP,
        vmin=-limit,
        vmax=limit,
    )
    axis.set_xticks(range(len(xlabels)), xlabels, rotation=45, ha="right")
    axis.set_yticks(range(len(ylabels)), ylabels)
    axis.set_title(
        title, loc="left", fontsize=R5F_PANEL_PT, fontweight="bold"
    )
    axis.tick_params(labelsize=R5F_TICK_PT, length=0)
    for spine in axis.spines.values():
        spine.set_visible(False)
    colorbar = axis.figure.colorbar(image, ax=axis, fraction=0.035, pad=0.02)
    colorbar.set_label(cbar_label, fontsize=R5F_TICK_PT)
    colorbar.ax.tick_params(labelsize=R5F_SMALL_PT)
    return image


def R5F_category_mapping(values, cmap_name, overrides=None):
    normalized = pd.Series(values).astype(object)
    normalized = normalized.where(normalized.notna(), "Unknown").astype(str)
    categories = sorted(normalized.unique())
    cmap = plt.get_cmap(cmap_name, max(len(categories), 1))
    mapping = {
        category: matplotlib.colors.to_hex(cmap(index))
        for index, category in enumerate(categories)
    }
    if overrides:
        mapping.update(
            {
                category: color
                for category, color in overrides.items()
                if category in categories
            }
        )
    if "Unknown" in mapping:
        mapping["Unknown"] = "#999999"
    return {category: mapping[category] for category in categories}


def R5F_rgb(values, mapping):
    normalized = pd.Series(values).astype(object)
    normalized = normalized.where(normalized.notna(), "Unknown").astype(str)
    return np.asarray(
        [matplotlib.colors.to_rgb(mapping[value]) for value in normalized]
    )


def R5F_metadata_legend(
    axis, groups, confidence_norm, activity_limit, compact=False
):
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.axis("off")
    legend_font = 4.1 if compact else R5F_SMALL_PT
    columns = [groups[:3], groups[3:]]
    for column_index, column_groups in enumerate(columns):
        x = 0.02 + 0.50 * column_index
        y = 0.99
        for title, mapping in column_groups:
            axis.text(
                x,
                y,
                title,
                fontsize=legend_font,
                fontweight="bold",
                va="top",
                transform=axis.transAxes,
            )
            y -= 0.050 if compact else 0.055
            for category_index, (label, color) in enumerate(mapping.items()):
                display_label = str(label)
                if compact:
                    if title == "Cell type":
                        display_label = display_label.replace(
                            "Epithelial ", ""
                        ).replace(" epithelial", "")
                    elif (
                        title == "Metastatic status"
                        and display_label == "Non-metastatic"
                    ):
                        display_label = "Non-met."
                    elif title == "Treatment":
                        display_label = f"T{category_index + 1}"
                    elif title == "Library":
                        display_label = display_label.replace("Library", "L")
                    R5F_G_COMPACT_ALIASES.setdefault(title, {})[
                        display_label
                    ] = str(label)
                axis.add_patch(
                    R5F_Rectangle(
                        (x, y - 0.011),
                        0.022,
                        0.022,
                        transform=axis.transAxes,
                        facecolor=color,
                        edgecolor="none",
                        clip_on=False,
                    )
                )
                axis.text(
                    x + 0.032,
                    y,
                    display_label,
                    fontsize=legend_font,
                    va="center",
                    transform=axis.transAxes,
                )
                y -= 0.034 if compact else 0.040
            y -= 0.018 if compact else 0.025

    confidence_cax = axis.inset_axes([0.02, 0.025, 0.42, 0.025])
    confidence_sm = matplotlib.cm.ScalarMappable(
        norm=confidence_norm, cmap="viridis"
    )
    confidence_bar = axis.figure.colorbar(
        confidence_sm, cax=confidence_cax, orientation="horizontal"
    )
    confidence_bar.set_label(
        "Confidence" if compact else "Cell-type confidence",
        fontsize=legend_font,
        labelpad=1,
    )
    confidence_bar.ax.tick_params(
        labelsize=legend_font, length=2, pad=1
    )

    activity_cax = axis.inset_axes([0.54, 0.025, 0.42, 0.025])
    activity_norm = matplotlib.colors.Normalize(
        vmin=-activity_limit, vmax=activity_limit
    )
    activity_sm = matplotlib.cm.ScalarMappable(
        norm=activity_norm, cmap=R5F_CMAP
    )
    activity_bar = axis.figure.colorbar(
        activity_sm, cax=activity_cax, orientation="horizontal"
    )
    activity_bar.set_label(
        "Activity z-score" if compact else "Cell-level activity z-score",
        fontsize=legend_font,
        labelpad=1,
    )
    activity_bar.ax.tick_params(
        labelsize=legend_font, length=2, pad=1
    )


def R5F_draw_B(axis):
    figure = axis.figure
    spec = axis.get_subplotspec()
    axis.remove()
    subgrid = spec.subgridspec(
        1, 2, width_ratios=[0.12, 1.0], wspace=0.035
    )
    group_axis = figure.add_subplot(subgrid[0, 0])
    heat_axis = figure.add_subplot(subgrid[0, 1])
    group_axis.imshow(
        R5_H_FACTOR_GROUPS[:, None],
        aspect="auto",
        interpolation="nearest",
        cmap=ListedColormap(CLUSTER_COLORS[:R5_PRIMARY_K]),
        vmin=0,
        vmax=R5_PRIMARY_K - 1,
    )
    group_axis.set_xticks([])
    group_axis.set_yticks(
        [R5_H_FACTOR_CENTERS[i] for i in range(R5_PRIMARY_K)],
        R5_COMPONENT_IDS,
        fontsize=R5F_SMALL_PT,
    )
    group_axis.set_ylabel("Component cluster", fontsize=R5F_TICK_PT)
    for spine in group_axis.spines.values():
        spine.set_visible(False)
    R5F_heatmap(
        heat_axis,
        R5_H_BY_CELLTYPE_GROUPED.to_numpy(),
        R5_CELLTYPE_ORDER,
        R5_H_FACTOR_LABELS,
        "B  H-factor activity by cell type",
        "Row z-score of log(1 + mean H)",
        98,
    )
    for boundary in R5_H_FACTOR_BOUNDARIES:
        heat_axis.axhline(boundary, color="white", linewidth=0.7)
        group_axis.axhline(boundary, color="white", linewidth=0.7)
    heat_axis.set_xlabel("Retained epithelial cell type")
    heat_axis.set_ylabel("Continuous H factors")
    heat_axis.tick_params(axis="y", labelsize=R5F_FACTOR_PT)


def R5F_draw_C(axis):
    R5F_heatmap(
        axis,
        R5_W_HEATMAP.to_numpy(),
        [str(value) for value in R5_W_HEATMAP.columns],
        [""] * len(R5_W_HEATMAP),
        "C  K=8 node-group definitions",
        "Node-wise W z-score",
        99,
    )
    axis.set_yticks(
        [R5_W_PROGRAM_CENTERS[i] for i in range(R5_PRIMARY_K)],
        R5_PROGRAM_IDS,
    )
    for boundary in R5_W_PROGRAM_BOUNDARIES:
        axis.axhline(boundary, color="white", linewidth=0.7)
    axis.set_xlabel("Component cluster")
    axis.set_ylabel("Decay-selected nodes")


def R5F_draw_supplementary_A(axis):
    figure = axis.figure
    spec = axis.get_subplotspec()
    axis.remove()
    R5F_draw_program_dictionary(
        figure=figure,
        parent_spec=spec,
        visible=R5W_REVIEW["visible_nodes"],
        factor_counts=R5_PROGRAM_FACTOR_COUNTS,
        category_colors=CATEGORY_COLORS,
        singleton_program=R5_SINGLETON_PROGRAM,
        singleton_factor=R5_SINGLETON_FACTOR,
        decay_tau=DECAY_TAU,
        decay_cumulative_mass=DECAY_CUMULATIVE_MASS,
        panel_label="A",
    )


def R5F_draw_D(axis):
    figure = axis.figure
    spec = axis.get_subplotspec()
    axis.remove()
    subgrid = spec.subgridspec(
        2,
        2,
        height_ratios=[0.18, 1.0],
        hspace=0.10,
        wspace=0.40,
    )
    title_axis = figure.add_subplot(subgrid[0, :])
    title_axis.axis("off")
    title_axis.text(
        0.0,
        0.95,
        "D  Program–component cluster–cell-type mappings",
        ha="left",
        va="top",
        fontsize=R5F_PANEL_PT,
        fontweight="bold",
        transform=title_axis.transAxes,
    )
    left_axis = figure.add_subplot(subgrid[1, 0])
    right_axis = figure.add_subplot(subgrid[1, 1])
    R5F_heatmap(
        left_axis,
        R5_PROGRAM_TO_COMPONENT.to_numpy(),
        R5_COMPONENT_IDS,
        R5_PROGRAM_IDS,
        "",
        "Mean W z-score",
    )
    left_axis.set_xlabel("Component cluster")
    left_axis.set_ylabel("Node group")
    R5F_heatmap(
        right_axis,
        R5_COMPONENT_BY_CELLTYPE_Z.to_numpy(),
        R5_CELLTYPE_ORDER,
        R5_COMPONENT_IDS,
        "",
        "Row z-score",
    )
    right_axis.set_xlabel("Retained epithelial cell type")


def R5F_curve(axis, x0, y0, x1, y1, color, width, alpha):
    path = R5F_Path(
        [
            (x0, y0),
            ((x0 + x1) / 2, y0),
            ((x0 + x1) / 2, y1),
            (x1, y1),
        ],
        [
            R5F_Path.MOVETO,
            R5F_Path.CURVE4,
            R5F_Path.CURVE4,
            R5F_Path.CURVE4,
        ],
    )
    axis.add_patch(
        R5F_PathPatch(
            path,
            facecolor="none",
            edgecolor=color,
            linewidth=width,
            alpha=alpha,
        )
    )


def R5F_draw_supplementary_G(axis):
    program_y = np.linspace(0.93, 0.07, R5_PRIMARY_K)
    component_y = np.linspace(0.93, 0.07, R5_PRIMARY_K)
    celltype_y = np.linspace(0.88, 0.12, len(R5_CELLTYPE_ORDER))
    program_to_component = R5_PROGRAM_TO_COMPONENT.to_numpy()
    component_to_celltype = R5_COMPONENT_BY_CELLTYPE_Z.to_numpy()

    for program in range(R5_PRIMARY_K):
        for component in range(R5_PRIMARY_K):
            strength = abs(program_to_component[program, component]) / max(
                np.abs(program_to_component).max(), 1e-12
            )
            R5F_curve(
                axis,
                0.10,
                program_y[program],
                0.47,
                component_y[component],
                "#B54A4A"
                if program_to_component[program, component] >= 0
                else "#4477AA",
                0.15 + 1.8 * strength,
                0.04 + 0.45 * strength,
            )
    for component in range(R5_PRIMARY_K):
        for celltype in range(len(R5_CELLTYPE_ORDER)):
            strength = abs(component_to_celltype[component, celltype]) / max(
                np.abs(component_to_celltype).max(), 1e-12
            )
            R5F_curve(
                axis,
                0.53,
                component_y[component],
                0.90,
                celltype_y[celltype],
                "#B54A4A"
                if component_to_celltype[component, celltype] >= 0
                else "#4477AA",
                0.15 + 1.8 * strength,
                0.04 + 0.45 * strength,
            )
    for index, label in enumerate(R5_PROGRAM_IDS):
        axis.scatter(
            0.08,
            program_y[index],
            s=42,
            color=CLUSTER_COLORS[index],
            edgecolor="white",
            linewidth=0.4,
            zorder=3,
        )
        axis.text(
            0.055,
            program_y[index],
            label,
            ha="right",
            va="center",
            fontsize=R5F_SMALL_PT,
        )
    for index, label in enumerate(R5_COMPONENT_IDS):
        axis.scatter(
            0.50,
            component_y[index],
            s=48,
            color="#D07A3D",
            edgecolor="white",
            linewidth=0.4,
            zorder=3,
        )
        axis.text(
            0.50,
            component_y[index],
            label,
            ha="center",
            va="center",
            fontsize=R5F_SMALL_PT,
            color="white",
            fontweight="bold",
        )
    for index, name in enumerate(R5_CELLTYPE_ORDER):
        axis.scatter(
            0.92,
            celltype_y[index],
            s=52,
            color=SUBTYPE_MAPPING[name],
            edgecolor="white",
            linewidth=0.4,
            zorder=3,
        )
        axis.text(
            0.94,
            celltype_y[index],
            name,
            ha="left",
            va="center",
            fontsize=R5F_SMALL_PT,
        )
    axis.set_xlim(0, 1.16)
    axis.set_ylim(0, 1.05)
    axis.axis("off")
    axis.set_title(
        "G  NG → CC → epithelial cell type",
        loc="left",
        fontsize=R5F_PANEL_PT,
        fontweight="bold",
        pad=12,
    )


def R5F_draw_F(axis):
    R5F_heatmap(
        axis,
        R5_PROGRAM_BY_CELLTYPE_Z.to_numpy(),
        R5_CELLTYPE_ORDER,
        R5_PROGRAM_IDS,
        "F  Reconstructed NG activity by cell type",
        "Row z-score",
    )
    axis.set_xlabel("Retained epithelial cell type")
    axis.set_ylabel("Node group")


R5F_G_METADATA_TRACK_NAMES = [
    "Final epithelial cell type",
    "Procedure/site",
    "Metastatic status",
    "Patient",
    "Treatment",
    "Library",
    "Cell-type confidence",
]
R5F_G_COMPACT_ALIASES = {}


def R5F_draw_G(axis):
    figure = axis.figure
    spec = axis.get_subplotspec()
    compact_legend = (
        spec.get_position(figure).width * figure.get_figwidth() < 7.5
    )
    axis.remove()
    subgrid = spec.subgridspec(
        8,
        2,
        height_ratios=[0.105] * 7 + [1.0],
        width_ratios=[4.4, 2.2],
        hspace=0.035,
        wspace=0.08,
    )

    celltype_mapping = {
        name: SUBTYPE_MAPPING[name] for name in R5_CELLTYPE_ORDER
    }
    site_mapping = R5F_category_mapping(
        R5_ORDERED_METADATA["Site"],
        "Set2",
        {
            "Bx": "#377EB8",
            "Surgery": "#E41A1C",
            "Recurrence": "#984EA3",
        },
    )
    metastatic_mapping = R5F_category_mapping(
        R5_ORDERED_METADATA["metastatic_status"],
        "Set2",
        {
            "Non-metastatic": "#4DAF4A",
            "Metastatic": "#E41A1C",
        },
    )
    patient_mapping = R5F_category_mapping(
        R5F_ORDERED_PATIENT_CODES, "tab10"
    )
    treatment_mapping = R5F_category_mapping(
        R5_ORDERED_METADATA["Treatment"], "Set2"
    )
    library_mapping = R5F_category_mapping(
        R5_ORDERED_METADATA["library"], "Set2"
    )
    categorical_tracks = [
        ("Final epithelial cell type", R5_ORDERED_CELLTYPES, celltype_mapping),
        ("Procedure/site", R5_ORDERED_METADATA["Site"], site_mapping),
        (
            "Metastatic status",
            R5_ORDERED_METADATA["metastatic_status"],
            metastatic_mapping,
        ),
        ("Patient", R5F_ORDERED_PATIENT_CODES, patient_mapping),
        ("Treatment", R5_ORDERED_METADATA["Treatment"], treatment_mapping),
        ("Library", R5_ORDERED_METADATA["library"], library_mapping),
    ]

    track_axes = []
    for row, (label, values, mapping) in enumerate(categorical_tracks):
        track_axis = figure.add_subplot(subgrid[row, 0])
        track_axis.imshow(
            R5F_rgb(values, mapping)[None, :, :],
            aspect="auto",
            interpolation="nearest",
        )
        track_axis.set_yticks([0], [label], fontsize=R5F_SMALL_PT)
        track_axis.set_xticks([])
        track_axis.tick_params(axis="y", length=0, pad=2)
        for boundary in R5_CELLTYPE_BOUNDARIES:
            track_axis.axvline(
                boundary - 0.5, color="white", linewidth=0.65
            )
        for spine in track_axis.spines.values():
            spine.set_visible(False)
        track_axes.append(track_axis)
    track_axes[0].set_title(
        "G  Cell-level reconstructed NG activity and metadata",
        loc="left",
        fontsize=R5F_PANEL_PT,
        fontweight="bold",
        pad=4,
    )

    confidence = pd.to_numeric(
        R5_ORDERED_METADATA["celltype_confidence"], errors="coerce"
    ).to_numpy(dtype=float)
    finite_confidence = confidence[np.isfinite(confidence)]
    if finite_confidence.size:
        confidence_low, confidence_high = np.nanpercentile(
            finite_confidence, [2, 98]
        )
        if confidence_high <= confidence_low:
            confidence_high = confidence_low + 1e-12
    else:
        confidence_low, confidence_high = 0.0, 1.0
    confidence_norm = matplotlib.colors.Normalize(
        vmin=confidence_low, vmax=confidence_high, clip=True
    )
    confidence_cmap = plt.get_cmap("viridis").copy()
    confidence_cmap.set_bad("#999999")
    confidence_axis = figure.add_subplot(subgrid[6, 0])
    confidence_axis.imshow(
        confidence_cmap(confidence_norm(confidence))[None, :, :],
        aspect="auto",
        interpolation="nearest",
    )
    confidence_axis.set_yticks(
        [0], ["Cell-type confidence"], fontsize=R5F_SMALL_PT
    )
    confidence_axis.set_xticks([])
    confidence_axis.tick_params(axis="y", length=0, pad=2)
    for boundary in R5_CELLTYPE_BOUNDARIES:
        confidence_axis.axvline(
            boundary - 0.5, color="white", linewidth=0.65
        )
    for spine in confidence_axis.spines.values():
        spine.set_visible(False)

    ordered = R5_PROGRAM_ACTIVITY_Z[R5_CELL_ORDER].T
    activity_limit = R5F_limit(ordered, 99)
    heat_axis = figure.add_subplot(subgrid[7, 0])
    heat_axis.imshow(
        ordered,
        aspect="auto",
        interpolation="nearest",
        cmap=R5F_CMAP,
        vmin=-activity_limit,
        vmax=activity_limit,
    )
    heat_axis.set_yticks(range(R5_PRIMARY_K), R5_PROGRAM_IDS)
    heat_axis.set_ylabel("Node group")
    for boundary in R5_CELLTYPE_BOUNDARIES:
        heat_axis.axvline(
            boundary - 0.5, color="white", linewidth=0.7
        )
    heat_axis.set_xticks([])
    heat_axis.set_xlabel(
        f"All epithelial cells (n={len(R5_CELL_ORDER):,})"
    )
    heat_axis.tick_params(
        axis="y", labelsize=R5F_TICK_PT, length=0
    )
    for spine in heat_axis.spines.values():
        spine.set_visible(False)

    legend_axis = figure.add_subplot(subgrid[:, 1])
    legend_groups = [
        ("Cell type", celltype_mapping),
        ("Site", site_mapping),
        ("Metastatic status", metastatic_mapping),
        ("Patient", patient_mapping),
        ("Treatment", treatment_mapping),
        ("Library", library_mapping),
    ]
    R5F_metadata_legend(
        legend_axis,
        legend_groups,
        confidence_norm,
        activity_limit,
        compact=compact_legend,
    )


R5F_MAIN_NODE_DEFINITIONS = pd.DataFrame([
    {"node_kind": "reaction", "node_name": "ACGAM2E",
     "display_label": "GlcNAc 2-epimerase reaction",
     "mechanistic_arm": "Predicted amino-sugar interconversion",
     "selection_rationale": "Top-ranked NG5* glycan reaction; recurrence-enriched in P3"},
    {"node_kind": "cellchat_entity", "node_name": "MMP14",
     "display_label": "MMP14 interaction node",
     "mechanistic_arm": "Predicted matrix-remodelling capacity",
     "selection_rationale": "NG5* matrix-interaction node; recurrence-enriched in P3"},
])
R5F_MAIN_NODE_NAMES = R5F_MAIN_NODE_DEFINITIONS["node_name"].tolist()
if not set(R5F_MAIN_NODE_NAMES).issubset(W.index):
    raise ValueError(f"Main mechanism nodes absent from W: {set(R5F_MAIN_NODE_NAMES) - set(W.index)}")
R5F_NG5_NODE_SET = set(
    MIXED_TOP_NODES.loc[MIXED_TOP_NODES["consensus_cluster"].eq(4), "node_name"].astype(str)
)
if not set(R5F_MAIN_NODE_NAMES).issubset(R5F_NG5_NODE_SET):
    raise ValueError("Main mechanism nodes must be selected NG5* nodes")
R5F_MAIN_NODE_RAW = H @ W.loc[R5F_MAIN_NODE_NAMES].to_numpy(dtype=float).T
R5F_MAIN_NODE_LOG = np.log1p(np.maximum(R5F_MAIN_NODE_RAW, 0))
R5F_MAIN_NODE_Z = (
    R5F_MAIN_NODE_LOG - R5F_MAIN_NODE_LOG.mean(axis=0, keepdims=True)
) / np.maximum(R5F_MAIN_NODE_LOG.std(axis=0, keepdims=True), 1e-12)
R5F_MAIN_NODE_FRAME = pd.DataFrame(R5F_MAIN_NODE_Z, columns=R5F_MAIN_NODE_NAMES)
R5F_MAIN_NODE_FRAME["Patient IDs"] = cell_metadata["Patient IDs"].astype(str).to_numpy()
R5F_MAIN_NODE_FRAME["Site"] = cell_metadata["Site"].astype(str).to_numpy()
R5F_CF30117_MAIN_NODE_FRAME = R5F_MAIN_NODE_FRAME[
    R5F_MAIN_NODE_FRAME["Patient IDs"].eq("P3")
].copy()
R5F_CF30117_MAIN_NODE_LONG = R5F_CF30117_MAIN_NODE_FRAME.melt(
    id_vars=["Patient IDs", "Site"], value_vars=R5F_MAIN_NODE_NAMES,
    var_name="node_name", value_name="activity_z",
)
R5F_MAIN_NODE_SUMMARY = (
    R5F_CF30117_MAIN_NODE_LONG.groupby(["node_name", "Site"], observed=True)["activity_z"]
    .agg(cells="size", mean_z="mean", median_z="median",
         q25_z=lambda values: values.quantile(0.25),
         q75_z=lambda values: values.quantile(0.75))
    .reset_index()
)
R5F_MAIN_NODE_SUMMARY = R5F_MAIN_NODE_SUMMARY.merge(
    R5F_MAIN_NODE_DEFINITIONS, on="node_name", how="left", validate="many_to_one"
)
R5F_MAIN_NODE_SUMMARY.to_csv(
    _derived_dir / "Figure_5_CF30117_main_node_violin_summary.csv", index=False
)


def R5F_draw_H(axis):
    figure = axis.figure
    spec = axis.get_subplotspec()
    axis.remove()
    subgrid = spec.subgridspec(1, 2, wspace=R5F_MAIN_NODE_WSPACE)
    for node_index, row in R5F_MAIN_NODE_DEFINITIONS.iterrows():
        target = figure.add_subplot(subgrid[0, node_index])
        groups = [
            R5F_CF30117_MAIN_NODE_FRAME.loc[
                R5F_CF30117_MAIN_NODE_FRAME["Site"].eq(site), row["node_name"]
            ].to_numpy()
            for site in R5F_SITE_ORDER
        ]
        parts = target.violinplot(
            groups, positions=range(len(R5F_SITE_ORDER)),
            showextrema=False, widths=R5F_VIOLIN_WIDTH,
        )
        for body, site in zip(parts["bodies"], R5F_SITE_ORDER):
            body.set_facecolor(R5F_SITE_COLORS[site])
            body.set_edgecolor("none")
            body.set_alpha(0.72)
        target.boxplot(
            groups, positions=range(len(R5F_SITE_ORDER)), widths=R5F_BOX_WIDTH,
            showfliers=False, patch_artist=True,
            boxprops={"facecolor": "white", "edgecolor": "#444444", "linewidth": 0.5},
            whiskerprops={"color": "#444444", "linewidth": 0.5},
            capprops={"color": "#444444", "linewidth": 0.5},
            medianprops={"color": "black", "linewidth": 0.8},
        )
        target.scatter(
            range(len(R5F_SITE_ORDER)), [np.mean(values) for values in groups],
            marker="D", s=R5F_MEAN_MARKER_SIZE, facecolor="black",
            edgecolor="white", linewidth=0.45, zorder=6, label="Mean",
        )
        counts = [f"{site}\nn={len(values):,}" for site, values in zip(R5F_SITE_ORDER, groups)]
        target.set_xticks(range(len(R5F_SITE_ORDER)), counts)
        target.set_ylabel("scKNIFE-inferred node activity\n(global z-score)" if node_index == 0 else "")
        target.set_title(
            ("H  " if node_index == 0 else "") + row["display_label"] +
            "\n" + row["mechanistic_arm"],
            loc="left", fontsize=R5F_AXIS_PT,
            fontweight="bold" if node_index == 0 else "normal",
        )
        target.axhline(0, color="#BBBBBB", linewidth=0.5, zorder=0)
        if node_index == 1:
            target.text(
                1.0, 1.02, "P3; descriptive cells from one longitudinal patient",
                transform=target.transAxes, ha="right", va="bottom",
                fontsize=R5F_SMALL_PT, color="#555555",
            )
        R5F_clean(target)


def R5F_draw_supplementary_E(axis):
    figure = axis.figure
    spec = axis.get_subplotspec()
    axis.remove()
    subgrid = spec.subgridspec(len(R5_SELECTED_FEATURES), 1, hspace=0.72)
    patient_ids = cell_metadata["Patient IDs"].astype(str).reset_index(drop=True)

    for feature_index, (_, _, label) in enumerate(R5_SELECTED_FEATURES):
        feature_axis = figure.add_subplot(subgrid[feature_index, 0])
        target_celltype = R5_FEATURE_RATIONALE.iloc[feature_index]["target_cell_type"]
        groups = [
            R5_FEATURE_Z[R5_CELLTYPE_INDEX.eq(celltype).to_numpy(), feature_index]
            for celltype in R5_CELLTYPE_ORDER
        ]
        parts = feature_axis.violinplot(
            groups, positions=range(len(R5_CELLTYPE_ORDER)),
            showextrema=False, widths=0.82,
        )
        for body, celltype in zip(parts["bodies"], R5_CELLTYPE_ORDER):
            body.set_facecolor(SUBTYPE_MAPPING[celltype])
            body.set_edgecolor("none")
            body.set_alpha(0.78 if celltype == target_celltype else 0.25)
        feature_axis.boxplot(
            groups, positions=range(len(R5_CELLTYPE_ORDER)), widths=0.14,
            showfliers=False, patch_artist=True,
            boxprops={"facecolor": "white", "edgecolor": "#444444", "linewidth": 0.4},
            whiskerprops={"color": "#444444", "linewidth": 0.4},
            capprops={"color": "#444444", "linewidth": 0.4},
            medianprops={"color": "black", "linewidth": 0.65},
        )
        for position, celltype in enumerate(R5_CELLTYPE_ORDER):
            mask = R5_CELLTYPE_INDEX.eq(celltype).to_numpy()
            patient_means = (
                pd.DataFrame({"patient": patient_ids[mask].to_numpy(),
                              "score": R5_FEATURE_Z[mask, feature_index]})
                .groupby("patient", observed=True)["score"].mean()
            )
            jitter = np.linspace(-0.055, 0.055, len(patient_means)) if len(patient_means) > 1 else np.array([0.0])
            feature_axis.scatter(
                position + jitter, patient_means.to_numpy(), s=R5F_E_PATIENT_DOT_SIZE,
                facecolor="white", edgecolor="#111111", linewidth=0.55,
                zorder=5, clip_on=True,
            )
        counts = [
            f"{value.replace('Epithelial ', '')}\nn={int(R5_CELLTYPE_INDEX.eq(value).sum()):,}, "
            f"p={int(patient_ids[R5_CELLTYPE_INDEX.eq(value)].nunique())}"
            for value in R5_CELLTYPE_ORDER
        ]
        feature_axis.set_xticks(range(len(R5_CELLTYPE_ORDER)))
        feature_axis.set_xticklabels(
            counts if feature_index == len(R5_SELECTED_FEATURES) - 1 else [],
            rotation=38, ha="right", fontsize=R5F_SMALL_PT,
        )
        feature_axis.set_ylabel("Inferred node score\n(z-score)" if feature_index == 2 else "")
        qualifier = " — weak association" if target_celltype == "Epithelial progenitor" else ""
        feature_axis.set_title(
            ("E  " if feature_index == 0 else "") + label + "\n" +
            f"Target: {target_celltype.replace('Epithelial ', '')}{qualifier}",
            loc="left", fontsize=R5F_AXIS_PT,
            fontweight="bold" if feature_index == 0 else "normal",
        )
        R5F_clean(feature_axis)


def R5F_draw_A(axis):
    figure, spec = axis.figure, axis.get_subplotspec(); axis.remove()
    full = R5W_REVIEW["full_nodes"]
    names = full["node_name"].astype(str).tolist()
    matrix = W_BY_COMPONENT_LOG_ZSCORE.loc[names].to_numpy(dtype=float)
    programs = full["program"].to_numpy(dtype=int)
    kinds = full["node_kind"].astype(str).to_numpy()
    kind_order = list(CATEGORY_COLORS)
    kind_codes = np.asarray([kind_order.index(kind) for kind in kinds], dtype=int)
    boundaries = np.flatnonzero(np.diff(programs) != 0) + 0.5
    centers = [float(np.mean(np.flatnonzero(programs == p))) for p in range(R5_PRIMARY_K)]
    sub = spec.subgridspec(2, 4, height_ratios=[0.35, 12], width_ratios=[6.5, .18, .28, 5.0], hspace=.04, wspace=.025)
    title = figure.add_subplot(sub[0, :]); title.axis("off")
    title.text(0, .95, "A  Full decay-selected W-node enrichment", ha="left", va="top", fontsize=R5F_PANEL_PT, fontweight="bold")
    present = [kind for kind in kind_order if kind in set(kinds)]
    title.legend(handles=[matplotlib.lines.Line2D([0],[0],marker="s",linestyle="none",markersize=3.5,markerfacecolor=CATEGORY_COLORS[k],markeredgecolor="none",label=k.replace("_"," ")) for k in present], loc="upper right", ncol=min(5,len(present)), frameon=False, fontsize=R5F_SMALL_PT, title="Node kind", title_fontsize=R5F_SMALL_PT, handletextpad=.25, columnspacing=.65)
    heat=figure.add_subplot(sub[1,0]); kind_ax=figure.add_subplot(sub[1,1],sharey=heat); prog_ax=figure.add_subplot(sub[1,2],sharey=heat); labels=figure.add_subplot(sub[1,3],sharey=heat)
    limit=R5F_limit(matrix,99)
    image=heat.imshow(matrix,aspect="auto",interpolation="nearest",cmap=R5F_CMAP,vmin=-limit,vmax=limit,rasterized=True)
    heat.set_xticks(range(R5_PRIMARY_K),R5_COMPONENT_IDS,rotation=45,ha="right"); heat.set_yticks([]); heat.set_xlabel("Component cluster",fontsize=R5F_TICK_PT); heat.set_ylabel(f"{len(full)} decay-selected nodes",fontsize=R5F_TICK_PT)
    kind_ax.imshow(kind_codes[:,None],aspect="auto",interpolation="nearest",cmap=ListedColormap([CATEGORY_COLORS[k] for k in kind_order]),vmin=0,vmax=max(len(kind_order)-1,1)); kind_ax.set_xticks([0],["Kind"],rotation=45,ha="right",fontsize=R5F_SMALL_PT); kind_ax.set_yticks([])
    prog_ax.imshow(programs[:,None],aspect="auto",interpolation="nearest",cmap=ListedColormap(CLUSTER_COLORS[:R5_PRIMARY_K]),vmin=0,vmax=R5_PRIMARY_K-1); prog_ax.set_xticks([0],["Node group"],rotation=45,ha="right",fontsize=R5F_SMALL_PT); prog_ax.set_yticks(centers,R5_PROGRAM_IDS,fontsize=R5F_SMALL_PT); prog_ax.tick_params(axis="y",length=0,pad=1)
    labels.set_xlim(0,1); labels.set_ylim(len(full)-.5,-.5); labels.axis("off")
    for row,name in enumerate(names): labels.text(0,row,name,ha="left",va="center",fontsize=3.1,color="#222222",clip_on=True)
    for boundary in boundaries:
        for target in (heat,kind_ax,prog_ax,labels): target.axhline(boundary,color="#333333",linewidth=.4)
    for target in (heat,kind_ax,prog_ax):
        for spine in target.spines.values(): spine.set_visible(False)
    cax=heat.inset_axes([0,-.090,1,.008]); cb=figure.colorbar(image,cax=cax,orientation="horizontal"); cb.set_label("Node-wise z-score of log(1 + mean W)",fontsize=R5F_SMALL_PT,labelpad=1); cb.ax.tick_params(labelsize=R5F_SMALL_PT,length=2,pad=1)


def R5F_draw_supplementary_B(axis):
    figure,spec=axis.figure,axis.get_subplotspec(); axis.remove()
    sub=spec.subgridspec(2,3,height_ratios=[.24,1],width_ratios=[1.2,1,1],hspace=.34,wspace=.42)
    title=figure.add_subplot(sub[0,:]); title.axis("off"); title.text(0,.95,"B  K=6 versus K=8 sensitivity",ha="left",va="top",fontsize=R5F_PANEL_PT,fontweight="bold")
    axes=[figure.add_subplot(sub[1,i]) for i in range(3)]; metrics=CONSENSUS_METRICS.set_index("k").loc[[6,8]]
    axes[0].bar(np.arange(2)-.18,metrics["PAC_0.1_0.9"],.36,color="#4477AA",label="PAC (lower)"); axes[0].bar(np.arange(2)+.18,1-metrics["consensus_silhouette"],.36,color="#CC6677",label="1 − silhouette (lower)"); axes[0].set_xticks(np.arange(2),["K=6","K=8"]); axes[0].set_title("Stability diagnostics",loc="left",fontsize=R5F_AXIS_PT); axes[0].legend(frameon=False,fontsize=R5F_SMALL_PT)
    for target,k in zip(axes[1:],[6,8]):
        counts=np.sort(np.bincount(np.asarray(CONSENSUS_SOLUTIONS[k]["labels"],dtype=int)))[::-1]; target.bar(np.arange(len(counts)),counts,color="#4477AA"); target.set_xticks(np.arange(len(counts)),[f"CC{i+1}" for i in range(len(counts))],rotation=45,ha="right"); target.set_ylabel("Latent factors"); target.set_title(f"K={k} group sizes",loc="left",fontsize=R5F_AXIS_PT)
    for target in axes: R5F_clean(target)


def R5F_draw_legacy_metabolic_supplement(axis):
    figure,spec=axis.figure,axis.get_subplotspec(); axis.remove()
    sub=spec.subgridspec(3,5,height_ratios=[.2,1,1],hspace=.78,wspace=.38)
    title=figure.add_subplot(sub[0,:]); title.axis("off"); title.text(0,.95,"C  Literature-prioritized metabolic graph-node scores",ha="left",va="top",fontsize=R5F_PANEL_PT,fontweight="bold")
    for i in range(len(R5_ALL_FEATURE_NAMES)):
        target=figure.add_subplot(sub[1+i//5,i%5]); groups=[R5_ALL_FEATURE_Z[R5_CELLTYPE_INDEX.eq(ct).to_numpy(),i] for ct in R5_CELLTYPE_ORDER]
        parts=target.violinplot(groups,positions=range(5),showextrema=False,widths=.82)
        for body,ct in zip(parts["bodies"],R5_CELLTYPE_ORDER): body.set_facecolor(SUBTYPE_MAPPING[ct]); body.set_alpha(.65); body.set_edgecolor("none")
        target.boxplot(groups,positions=range(5),widths=.14,showfliers=False,patch_artist=True,boxprops=dict(facecolor="white",linewidth=.4),medianprops=dict(color="black",linewidth=.65)); target.set_title(R5_FEATURE_RATIONALE.iloc[i]["display_label"],fontsize=R5F_SMALL_PT); target.set_xticks(range(5),[v.replace("Epithelial ","") for v in R5_CELLTYPE_ORDER],rotation=45,ha="right",fontsize=4.2)
        if i%5==0: target.set_ylabel("Inferred score (z)",fontsize=R5F_SMALL_PT)
        R5F_clean(target)


R5F_H_PROGRAM_CELL = np.column_stack([
    np.asarray(H)[:, COMPONENT_ASSIGNMENTS.loc[
        COMPONENT_ASSIGNMENTS["consensus_cluster"].eq(cluster), "component"
    ].astype(int).to_numpy()].mean(axis=1)
    for cluster in range(R5_PRIMARY_K)
])
R5F_H_PROGRAM_CELL_FRAME = pd.DataFrame(R5F_H_PROGRAM_CELL, columns=R5_PROGRAM_IDS)
R5F_H_PROGRAM_CELL_FRAME["Patient IDs"] = cell_metadata["Patient IDs"].astype(str).to_numpy()
R5F_PATIENT_H_PROGRAM_MEAN = (
    R5F_H_PROGRAM_CELL_FRAME.groupby("Patient IDs", observed=True)[R5_PROGRAM_IDS]
    .mean().reindex(R5F_PATIENT_ORDER)
)
R5F_PATIENT_H_PROGRAM_Z = (
    R5F_PATIENT_H_PROGRAM_MEAN - R5F_PATIENT_H_PROGRAM_MEAN.mean(axis=0)
) / R5F_PATIENT_H_PROGRAM_MEAN.std(axis=0, ddof=0).replace(0, np.nan)
if R5F_PATIENT_H_PROGRAM_Z.isna().any().any():
    raise RuntimeError("Patient-by-program standardization produced missing values")
R5F_REFERENCE_PATIENT_Z_PATH = NOTEBOOK_DIR / "data/Figure_5_patient_heterogeneity/derived/patient_program_scores_column_z.csv"
if R5F_REFERENCE_PATIENT_Z_PATH.exists():
    R5F_REFERENCE_PATIENT_Z = pd.read_csv(R5F_REFERENCE_PATIENT_Z_PATH).set_index("Patient IDs")
    R5F_REFERENCE_PATIENT_Z = R5F_REFERENCE_PATIENT_Z.rename(columns={"NG5": "NG5*"}).reindex(index=R5F_PATIENT_ORDER, columns=R5_PROGRAM_IDS)
    if not np.allclose(R5F_PATIENT_H_PROGRAM_Z.to_numpy(), R5F_REFERENCE_PATIENT_Z.to_numpy(), atol=1e-10):
        raise RuntimeError("Figure 5 patient-program scores do not reproduce the reference analysis")
R5F_PATIENT_H_PROGRAM_EXPORT = pd.DataFrame({
    "display_id": R5F_PATIENT_DISPLAY_ORDER,
    "Patient IDs": R5F_PATIENT_ORDER,
})
for program in R5_PROGRAM_IDS:
    R5F_PATIENT_H_PROGRAM_EXPORT[f"{program}_mean"] = R5F_PATIENT_H_PROGRAM_MEAN[program].to_numpy()
    R5F_PATIENT_H_PROGRAM_EXPORT[f"{program}_z"] = R5F_PATIENT_H_PROGRAM_Z[program].to_numpy()
R5F_PATIENT_H_PROGRAM_EXPORT.to_csv(
    _derived_dir / "Figure_5_patient_H_program_means_and_z.csv", index=False
)


def R5F_draw_supplementary_F(axis):
    matrix = R5F_PATIENT_H_PROGRAM_Z.to_numpy()
    limit = R5F_limit(matrix, 98)
    mesh = axis.pcolormesh(
        matrix, cmap=R5F_CMAP, vmin=-limit, vmax=limit,
        shading="flat", edgecolors="white", linewidth=0.55,
    )
    axis.set_xlim(0, len(R5_PROGRAM_IDS))
    axis.set_ylim(len(R5F_PATIENT_DISPLAY_ORDER), 0)
    axis.set_xticks(np.arange(len(R5_PROGRAM_IDS)) + 0.5, R5_PROGRAM_IDS)
    axis.set_yticks(np.arange(len(R5F_PATIENT_DISPLAY_ORDER)) + 0.5, R5F_PATIENT_DISPLAY_ORDER)
    axis.set_xlabel("H-factor-defined epithelial program")
    axis.set_ylabel("Patient")
    axis.set_title("F  Patient-average epithelial program activity", loc="left", fontsize=R5F_PANEL_PT, fontweight="bold")
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = matrix[row, col]
            axis.text(col + 0.5, row + 0.5, f"{value:.2f}", ha="center", va="center", fontsize=R5F_TICK_PT, color="white" if abs(value) > 0.58 * limit else "#222222")
    colorbar = axis.figure.colorbar(mesh, ax=axis, fraction=0.035, pad=0.025)
    colorbar.set_label("Patient mean program activity (column z-score)", fontsize=R5F_SMALL_PT)
    colorbar.ax.tick_params(labelsize=R5F_SMALL_PT)
    axis.text(1.0, -0.24, "Cell-weighted patient means; descriptive across seven patients", transform=axis.transAxes, ha="right", va="top", fontsize=R5F_SMALL_PT, color="#555555")
    R5F_clean(axis)


R5F_CONNECTION_X = {"NG": 0.055, "CC": 0.315, "cell_type": 0.585, "patient": 0.915}
R5F_CONNECTION_ASSOC_WIDTH_MIN = 0.12
R5F_CONNECTION_ASSOC_WIDTH_RANGE = 1.35
R5F_CONNECTION_COUNT_WIDTH_MIN = 0.20
R5F_CONNECTION_COUNT_WIDTH_RANGE = 2.80
R5F_CONNECTION_ASSOC_ALPHA_MIN = 0.035
R5F_CONNECTION_ASSOC_ALPHA_RANGE = 0.38
R5F_CONNECTION_COUNT_ALPHA = 0.58
R5F_CONNECTION_NODE_SIZE = 58
R5F_CONNECTION_COUNT_FONT_PT = 4.2
R5F_CONNECTION_PATIENT_COLORS = dict(zip(
    R5F_PATIENT_DISPLAY_ORDER,
    [matplotlib.colors.to_hex(color) for color in plt.get_cmap("Set2")(np.linspace(0, 1, len(R5F_PATIENT_DISPLAY_ORDER)))]
))
R5F_CONNECTION_CELLTYPE_TOTALS = R5F_PATIENT_CELLTYPE_COUNTS.sum(axis=0).reindex(R5_CELLTYPE_ORDER)
R5F_CONNECTION_PATIENT_TOTALS = R5F_PATIENT_CELLTYPE_COUNTS.sum(axis=1).reindex(R5F_PATIENT_ORDER)
if int(R5F_CONNECTION_CELLTYPE_TOTALS.sum()) != len(cell_metadata) or int(R5F_CONNECTION_PATIENT_TOTALS.sum()) != len(cell_metadata):
    raise RuntimeError("Patient/cell-type census does not sum to the retained epithelial-cell total")
R5F_CONNECTION_MAX_COUNT = int(R5F_PATIENT_CELLTYPE_COUNTS.to_numpy().max())
R5F_CONNECTION_EDGE_ROWS = []
R5F_P2C = R5_PROGRAM_TO_COMPONENT.to_numpy(dtype=float)
R5F_C2T = R5_COMPONENT_BY_CELLTYPE_Z.to_numpy(dtype=float)
R5F_P2C_MAX = max(float(np.abs(R5F_P2C).max()), 1e-12)
R5F_C2T_MAX = max(float(np.abs(R5F_C2T).max()), 1e-12)
for source_index, source in enumerate(R5_PROGRAM_IDS):
    for target_index, target in enumerate(R5_COMPONENT_IDS):
        value = float(R5F_P2C[source_index, target_index])
        strength = abs(value) / R5F_P2C_MAX
        R5F_CONNECTION_EDGE_ROWS.append({"edge_type": "NG_to_CC_association", "source": source, "target": target, "raw_value": value, "sign": "positive" if value >= 0 else "negative", "cell_count": np.nan, "plot_width": R5F_CONNECTION_ASSOC_WIDTH_MIN + R5F_CONNECTION_ASSOC_WIDTH_RANGE * strength})
for source_index, source in enumerate(R5_COMPONENT_IDS):
    for target_index, target in enumerate(R5_CELLTYPE_ORDER):
        value = float(R5F_C2T[source_index, target_index])
        strength = abs(value) / R5F_C2T_MAX
        R5F_CONNECTION_EDGE_ROWS.append({"edge_type": "CC_to_celltype_association", "source": source, "target": target, "raw_value": value, "sign": "positive" if value >= 0 else "negative", "cell_count": np.nan, "plot_width": R5F_CONNECTION_ASSOC_WIDTH_MIN + R5F_CONNECTION_ASSOC_WIDTH_RANGE * strength})
for cell_type in R5_CELLTYPE_ORDER:
    for patient_id, display_id in zip(R5F_PATIENT_ORDER, R5F_PATIENT_DISPLAY_ORDER):
        count = int(R5F_PATIENT_CELLTYPE_COUNTS.loc[patient_id, cell_type])
        if count > 0:
            strength = np.log10(count + 1) / np.log10(R5F_CONNECTION_MAX_COUNT + 1)
            R5F_CONNECTION_EDGE_ROWS.append({"edge_type": "celltype_to_patient_count", "source": cell_type, "target": display_id, "raw_value": count, "sign": "observed_count", "cell_count": count, "plot_width": R5F_CONNECTION_COUNT_WIDTH_MIN + R5F_CONNECTION_COUNT_WIDTH_RANGE * strength})
R5F_CONNECTION_EDGES = pd.DataFrame(R5F_CONNECTION_EDGE_ROWS)
if len(R5F_CONNECTION_EDGES.query("edge_type == 'celltype_to_patient_count'")) != 25:
    raise RuntimeError("Expected 25 non-zero cell-type-to-patient links")
R5F_CONNECTION_EDGES.to_csv(_derived_dir / "Figure_5_NG_CC_celltype_patient_edges.csv", index=False)


def R5F_connection_bezier_y(y0, y1, fraction):
    return y0 * ((1 - fraction) ** 3 + 3 * (1 - fraction) ** 2 * fraction) + y1 * (3 * (1 - fraction) * fraction ** 2 + fraction ** 3)


def R5F_draw_E(axis):
    program_y = np.linspace(0.88, 0.18, len(R5_PROGRAM_IDS))
    component_y = np.linspace(0.88, 0.18, len(R5_COMPONENT_IDS))
    celltype_y = np.linspace(0.84, 0.22, len(R5_CELLTYPE_ORDER))
    patient_y = np.linspace(0.88, 0.18, len(R5F_PATIENT_DISPLAY_ORDER))
    for program_index in range(len(R5_PROGRAM_IDS)):
        for component_index in range(len(R5_COMPONENT_IDS)):
            value = R5F_P2C[program_index, component_index]
            strength = abs(value) / R5F_P2C_MAX
            R5F_curve(axis, R5F_CONNECTION_X["NG"], program_y[program_index], R5F_CONNECTION_X["CC"], component_y[component_index], "#B54A4A" if value >= 0 else "#4477AA", R5F_CONNECTION_ASSOC_WIDTH_MIN + R5F_CONNECTION_ASSOC_WIDTH_RANGE * strength, R5F_CONNECTION_ASSOC_ALPHA_MIN + R5F_CONNECTION_ASSOC_ALPHA_RANGE * strength)
    for component_index in range(len(R5_COMPONENT_IDS)):
        for celltype_index in range(len(R5_CELLTYPE_ORDER)):
            value = R5F_C2T[component_index, celltype_index]
            strength = abs(value) / R5F_C2T_MAX
            R5F_curve(axis, R5F_CONNECTION_X["CC"], component_y[component_index], R5F_CONNECTION_X["cell_type"], celltype_y[celltype_index], "#B54A4A" if value >= 0 else "#4477AA", R5F_CONNECTION_ASSOC_WIDTH_MIN + R5F_CONNECTION_ASSOC_WIDTH_RANGE * strength, R5F_CONNECTION_ASSOC_ALPHA_MIN + R5F_CONNECTION_ASSOC_ALPHA_RANGE * strength)
    for celltype_index, cell_type in enumerate(R5_CELLTYPE_ORDER):
        for patient_index, (patient_id, display_id) in enumerate(zip(R5F_PATIENT_ORDER, R5F_PATIENT_DISPLAY_ORDER)):
            count = int(R5F_PATIENT_CELLTYPE_COUNTS.loc[patient_id, cell_type])
            if count == 0:
                continue
            strength = np.log10(count + 1) / np.log10(R5F_CONNECTION_MAX_COUNT + 1)
            width = R5F_CONNECTION_COUNT_WIDTH_MIN + R5F_CONNECTION_COUNT_WIDTH_RANGE * strength
            R5F_curve(axis, R5F_CONNECTION_X["cell_type"], celltype_y[celltype_index], R5F_CONNECTION_X["patient"], patient_y[patient_index], R5F_CONNECTION_PATIENT_COLORS[display_id], width, R5F_CONNECTION_COUNT_ALPHA)
            label_fraction = 0.69
            label_y = R5F_connection_bezier_y(celltype_y[celltype_index], patient_y[patient_index], label_fraction)
            label_x = R5F_CONNECTION_X["cell_type"] + label_fraction * (R5F_CONNECTION_X["patient"] - R5F_CONNECTION_X["cell_type"])
            axis.text(label_x, label_y, f"{count:,}", ha="center", va="center", fontsize=R5F_CONNECTION_COUNT_FONT_PT, color="#222222", zorder=6, bbox={"boxstyle": "round,pad=0.10", "facecolor": "white", "edgecolor": "none", "alpha": 0.78})
    for index, label in enumerate(R5_PROGRAM_IDS):
        axis.scatter(R5F_CONNECTION_X["NG"], program_y[index], s=R5F_CONNECTION_NODE_SIZE, color=CLUSTER_COLORS[index], edgecolor="white", linewidth=0.45, zorder=4)
        axis.text(R5F_CONNECTION_X["NG"] - 0.018, program_y[index], label, ha="right", va="center", fontsize=R5F_SMALL_PT)
    for index, label in enumerate(R5_COMPONENT_IDS):
        axis.scatter(R5F_CONNECTION_X["CC"], component_y[index], s=R5F_CONNECTION_NODE_SIZE, color="#D07A3D", edgecolor="white", linewidth=0.45, zorder=4)
        axis.text(R5F_CONNECTION_X["CC"], component_y[index], label, ha="center", va="center", fontsize=R5F_SMALL_PT, color="white", fontweight="bold", zorder=5)
    short_celltype_labels = [value.replace("Epithelial ", "").replace(" epithelial", "") for value in R5_CELLTYPE_ORDER]
    for index, (name, short_label) in enumerate(zip(R5_CELLTYPE_ORDER, short_celltype_labels)):
        axis.scatter(R5F_CONNECTION_X["cell_type"], celltype_y[index], s=R5F_CONNECTION_NODE_SIZE, color=SUBTYPE_MAPPING[name], edgecolor="white", linewidth=0.45, zorder=4)
        axis.text(R5F_CONNECTION_X["cell_type"] + 0.014, celltype_y[index] + 0.022, f"{short_label}\nn={int(R5F_CONNECTION_CELLTYPE_TOTALS[name]):,}", ha="left", va="bottom", fontsize=R5F_SMALL_PT, zorder=5)
    for index, (patient_id, display_id) in enumerate(zip(R5F_PATIENT_ORDER, R5F_PATIENT_DISPLAY_ORDER)):
        axis.scatter(R5F_CONNECTION_X["patient"], patient_y[index], s=R5F_CONNECTION_NODE_SIZE, color=R5F_CONNECTION_PATIENT_COLORS[display_id], edgecolor="white", linewidth=0.45, zorder=4)
        axis.text(R5F_CONNECTION_X["patient"] + 0.018, patient_y[index], f"{display_id}  n={int(R5F_CONNECTION_PATIENT_TOTALS[patient_id]):,}", ha="left", va="center", fontsize=R5F_SMALL_PT, fontweight="bold")
    for key, title in [("NG", "Node group"), ("CC", "Component cluster"), ("cell_type", "Epithelial cell type"), ("patient", "Patient")]:
        axis.text(R5F_CONNECTION_X[key], 0.965, title, ha="center", va="bottom", fontsize=R5F_AXIS_PT, fontweight="bold")
    axis.plot([0.08, 0.12], [0.075, 0.075], color="#B54A4A", linewidth=1.4, alpha=0.8)
    axis.text(0.125, 0.075, "positive model association", va="center", fontsize=R5F_SMALL_PT)
    axis.plot([0.30, 0.34], [0.075, 0.075], color="#4477AA", linewidth=1.4, alpha=0.8)
    axis.text(0.345, 0.075, "negative model association", va="center", fontsize=R5F_SMALL_PT)
    axis.plot([0.55, 0.59], [0.075, 0.075], color=R5F_CONNECTION_PATIENT_COLORS["P3"], linewidth=2.3, alpha=R5F_CONNECTION_COUNT_ALPHA)
    axis.text(0.595, 0.075, "observed cell membership; width = log10(n + 1), labels = exact n", va="center", fontsize=R5F_SMALL_PT)
    axis.text(0.5, 0.025, "Association layers are not literal cell flow; cell counts enter only at cell type → patient.", ha="center", va="center", fontsize=R5F_SMALL_PT, color="#444444", fontstyle="italic")
    axis.set_xlim(0, 1.07)
    axis.set_ylim(0, 1.03)
    axis.axis("off")
    axis.set_title("E  NG → CC → epithelial cell type → patient", loc="left", fontsize=R5F_PANEL_PT, fontweight="bold", pad=10)


R5F_DRAWERS={"A":R5F_draw_A,"B":R5F_draw_B,"C":R5F_draw_C,"D":R5F_draw_D,"E":R5F_draw_E,"F":R5F_draw_F,"G":R5F_draw_G,"H":R5F_draw_H}
R5F_SUPPLEMENTARY_DRAWERS={"B":R5F_draw_supplementary_B,"D":R5F_draw_supplementary_D,"E":R5F_draw_supplementary_E,"F":R5F_draw_supplementary_F,"G":R5F_draw_supplementary_G}
R5F_SUPPLEMENTARY_CONTENT={"B":"K=6 versus K=8 consensus sensitivity","D":"P1–P7 representation and equal-patient sensitivity","E":"Five literature-prioritized functional non-gene representatives","F":"Patient-level reconstructed NG activity sensitivity","G":"Cell-level program activity with P1–P7 metadata tracks"}
R5F_SIZES={"A":(9.2,16.3),"B":(8,9),"C":(8,10),"D":(14,5),"E":(16,8.5),"F":(8,4),"G":(12,7),"H":(12,4.2)}
R5F_SUPPLEMENTARY_SIZES={"A":(8,8),"B":(8,2.1),"C":(14,6),"D":(8,2.2),"E":(8,5.6),"F":(8,3),"G":(10,4)}
R5F_E_PATIENT_DOT_SIZE = 9.5
R5F_PANEL_CONTENT={"A":"Full decay-selected W-node enrichment heatmap","B":"H-factor activity by epithelial cell type, grouped by factor cluster","C":"K=8 W-node definitions grouped by node group","D":"NG to CC to epithelial-cell-type heatmaps","E":"NG to CC to epithelial-cell-type associations extended to observed P1–P7 cell counts","F":"Reconstructed NG activity by cell type","G":"Cell-level program activity with P1–P7 metadata tracks","H":"P3 longitudinal ACGAM2E and MMP14 inferred-node activities"}

def R5F_save(figure,base,tight=True):
    opts={"bbox_inches":"tight"} if tight else {}
    figure.savefig(base.with_suffix(".png"),dpi=R5F_DPI,facecolor="white",**opts)
    with matplotlib.rc_context(R5F_VECTOR_TEXT_RCPARAMS):
        figure.savefig(base.with_suffix(".pdf"),facecolor="white",**opts); figure.savefig(base.with_suffix(".svg"),facecolor="white",**opts)

R5F_DATA_MANIFEST=json.loads((NOTEBOOK_DIR/"data/Figure_5/manifest.json").read_text()); R5F_PANEL_LINEAGE=R5F_DATA_MANIFEST["panels"]
vector_metadata={"font_family":R5F_FONT_FAMILY,"pdf_font_type":R5F_PDF_FONT_TYPE,"pdf_text_mode":"editable_truetype","svg_font_type":R5F_SVG_FONT_TYPE,"svg_text_mode":"editable_text"}
allowed={f"Figure_5_{kind}_panel.{suffix}" for kind in ("main","supplementary") for suffix in ("png","pdf","svg")}|{"Figure_5_main_panel_manifest.json","Figure_5_supplementary_panel_manifest.json"}
for path in _compiled_dir.iterdir():
    if path.is_file() and path.name not in allowed: path.unlink()

manifest={}
for letter in "ACEGH":
    drawer=R5F_DRAWERS[letter]
    # Export only panels retained in the main composition.
    fig,ax=plt.subplots(figsize=R5F_SIZES[letter],constrained_layout=False); drawer(ax); fig.subplots_adjust(left=.08 if letter=="A" else .11,right=.97 if letter=="A" else .91,bottom=.06 if letter=="A" else .17,top=.97 if letter=="A" else .90,hspace=.58,wspace=.42); R5F_save(fig,_individual_dir/f"Figure_5_{letter}",True); plt.show(); plt.close(fig)
    lineage=R5F_PANEL_LINEAGE[letter]; manifest[letter]={"revision":R5F_REVISION,"grouping":"P3 Site" if letter=="H" else "scknife_final_celltype","content":R5F_PANEL_CONTENT[letter],"formats":["png","pdf","svg"],"mode":"standalone_data_driven_figure","renderer":"matplotlib_fallback","matplotlib_version":matplotlib.__version__,**vector_metadata,"svg_raster_images":None,"saved_in_individual_notebook_cell":True,"compiled_canvas_crop":False,"fixed_canvas_export":False,"export_bounds":"content_aware_tight","metadata_tracks":R5F_G_METADATA_TRACK_NAMES if letter=="G" else [],"data_lineage_kind":lineage["kind"],"intermediate_inputs":lineage["inputs"],"data_extractor":lineage["extractor"],"extraction_mode":lineage["extraction_mode"]}

supp_manifest={}
for letter,drawer in R5F_SUPPLEMENTARY_DRAWERS.items():
    fig,ax=plt.subplots(figsize=R5F_SUPPLEMENTARY_SIZES[letter],constrained_layout=False); drawer(ax); fig.subplots_adjust(left=.07,right=.96,bottom=.10,top=.94,hspace=.35,wspace=.30); R5F_save(fig,_individual_dir/f"Figure_5_supplementary_{letter}",True); plt.show(); plt.close(fig)
    supp_grouping = "scknife_final_celltype" if letter in {"E", "G"} else ("patient" if letter == "F" else None)
    supp_manifest[letter]={"revision":R5F_REVISION,"grouping":supp_grouping,"content":R5F_SUPPLEMENTARY_CONTENT[letter],"formats":["png","pdf","svg"],"mode":"standalone_data_driven_figure","renderer":"matplotlib_fallback","matplotlib_version":matplotlib.__version__,**vector_metadata,"svg_raster_images":None,"saved_in_individual_notebook_cell":True,"compiled_canvas_crop":False,"fixed_canvas_export":False,"export_bounds":"content_aware_tight"}

fig=plt.figure(figsize=(R5F_WIDTH_MM/25.4,R5F_HEIGHT_MM/25.4),constrained_layout=False); grid=fig.add_gridspec(5,2,height_ratios=(1.0,1.0,1.0,1.0,1.0),left=R5F_MAIN_LEFT,right=R5F_MAIN_RIGHT,bottom=R5F_MAIN_BOTTOM,top=R5F_MAIN_TOP,hspace=R5F_MAIN_HSPACE,wspace=R5F_MAIN_WSPACE)
specs={"A":grid[0,:],"C":grid[1,:],"E":grid[2,:],"G":grid[3,:],"H":grid[4,:]}
for letter in "ACEGH": R5F_DRAWERS[letter](fig.add_subplot(specs[letter]))
fig.suptitle("TNBC epithelial H→node groups after annotation-node exclusion",fontsize=12,fontweight="bold",y=.995); fig.supxlabel("Cell-type and cell-state graph nodes are excluded before W clustering; epithelial cell types remain biological activity groupings.",fontsize=R5F_SMALL_PT,color="#444444",y=.006); R5F_save(fig,_compiled_dir/"Figure_5_main_panel",False); plt.show(); plt.close(fig)

supp=plt.figure(figsize=(R5F_WIDTH_MM/25.4,R5F_SUPPLEMENT_HEIGHT_MM/25.4),constrained_layout=False); sg=supp.add_gridspec(3,1,height_ratios=(0.72,1.55,1.0),left=.055,right=.965,bottom=.045,top=.935,hspace=.38)
sg_top=sg[0,0].subgridspec(1,2,width_ratios=(1.0,1.0),wspace=.30)
sg_middle=sg[1,0].subgridspec(1,2,width_ratios=(0.40,0.60),wspace=.28)
R5F_SUPPLEMENTARY_DRAWERS["B"](supp.add_subplot(sg_top[0,0]))
R5F_SUPPLEMENTARY_DRAWERS["D"](supp.add_subplot(sg_top[0,1]))
R5F_SUPPLEMENTARY_DRAWERS["E"](supp.add_subplot(sg_middle[0,0]))
R5F_SUPPLEMENTARY_DRAWERS["F"](supp.add_subplot(sg_middle[0,1]))
R5F_SUPPLEMENTARY_DRAWERS["G"](supp.add_subplot(sg[2,0]))
supp.suptitle("Figure 5 supplementary analyses",fontsize=11,fontweight="bold",y=.975); R5F_save(supp,_compiled_dir/"Figure_5_supplementary_panel",False); plt.show(); plt.close(supp)

CONSENSUS_METRICS.to_csv(_derived_dir/"Figure_5_consensus_metrics.csv",index=False); COMPONENT_ASSIGNMENTS.to_csv(_derived_dir/"Figure_5_component_assignments.csv",index=False)
patient_id_metadata={"source":str(R5F_PATIENT_ID_MAPPING_PATH.relative_to(NOTEBOOK_DIR)),"visual_labels":"P1-P7","mapping":R5F_PATIENT_ID_TABLE[["display_id","patient_id"]].to_dict(orient="records")}
connection_panel_metadata={"panel":"Main E","layers":["NG","CC","epithelial cell type","patient"],"association_edge_count":int((R5F_CONNECTION_EDGES["edge_type"] != "celltype_to_patient_count").sum()),"observed_count_edge_count":int((R5F_CONNECTION_EDGES["edge_type"] == "celltype_to_patient_count").sum()),"observed_cell_total":int(R5F_CONNECTION_PATIENT_TOTALS.sum()),"count_width_transform":"log10(n + 1)","count_labels":"exact n","interpretation":"association layers are not literal cell flow; observed cell counts enter only at cell type to patient","inferential_tests":False}
supplementary_association_flow_metadata={"panel":"Supplementary G","layers":["NG","CC","epithelial cell type"],"association_edge_count":int((R5F_CONNECTION_EDGES["edge_type"] != "celltype_to_patient_count").sum()),"interpretation":"signed model associations only; no patient cell-count layer","inferential_tests":False}
run={"revision":R5F_REVISION,"random_seed":SEED,"selected_k":int(SELECTED_K),"raw_w_nodes":int(len(W)),"excluded_node_kinds":list(EXCLUDED_W_NODE_KINDS),"excluded_node_kind_counts":{key:int(value) for key,value in EXCLUDED_NODE_KIND_COUNTS.items()},"eligible_w_nodes":int(len(W_ELIGIBLE)),"selected_nodes":int(len(MIXED_TOP_NODES)),"excluded_annotation_nodes_absent":not MIXED_TOP_NODES["node_kind"].astype(str).isin(EXCLUDED_W_NODE_KINDS).any(),"program_alignment":PROGRAM_ALIGNMENT.to_dict(orient="records"),"analysis_grouping":"scknife_final_celltype","patient_id_metadata":patient_id_metadata,"patient_program_panel":{"panel":"Supplementary F","cell_score":"mean assigned H factors","patient_score":"cell-weighted mean","display_scale":"column z-score across seven patients; ddof=0"},"connection_panel":connection_panel_metadata,"violin_presentation":"descriptive cell-level distributions; no inferential tests","vector_text":vector_metadata}; (_derived_dir/"Figure_5_run_manifest.json").write_text(json.dumps(run,indent=2)+"\n")
individual={"revision":R5F_REVISION,"renderer":R5F_RENDERER,"vector_text":vector_metadata,"patient_id_metadata":patient_id_metadata,"connection_panel":connection_panel_metadata,"supplementary_association_flow":supplementary_association_flow_metadata,"main_panel_order":list("ACEGH"),"supplementary_panel_order":list("BDEFG"),"violin_presentation":{"main_panel":"H","main_grouping":"P3 Site","supplementary_panel":"E","supplementary_grouping":"scknife_final_celltype","interpretation":"descriptive cell-level distributions; cells are not independent biological replicates","inferential_tests":False},"panels":manifest,"supplementary_panels":supp_manifest}; (_individual_dir/"Figure_5_panel_export_manifest.json").write_text(json.dumps(individual,indent=2)+"\n")
main_m={"revision":R5F_REVISION,"renderer":R5F_RENDERER,"vector_text":vector_metadata,"patient_id_metadata":patient_id_metadata,"connection_panel":connection_panel_metadata,"canvas_mm":{"width":R5F_WIDTH_MM,"height":R5F_HEIGHT_MM},"panel_order":list("ACEGH"),"violin_presentation":{"panel":"H","grouping":"P3 Site","nodes":R5F_MAIN_NODE_NAMES,"interpretation":"descriptive cell-level distributions from one longitudinal patient","inferential_tests":False},"panels":manifest,"program_names_in_main":False,"neutral_program_ids_in_main":True,"excluded_node_kinds":list(EXCLUDED_W_NODE_KINDS)}; (_compiled_dir/"Figure_5_main_panel_manifest.json").write_text(json.dumps(main_m,indent=2)+"\n")
supp_m={"revision":R5F_REVISION,"renderer":R5F_RENDERER,"vector_text":vector_metadata,"patient_id_metadata":patient_id_metadata,"canvas_mm":{"width":R5F_WIDTH_MM,"height":R5F_SUPPLEMENT_HEIGHT_MM},"panel_order":list("BDEFG"),"violin_presentation":{"panel":"E","grouping":"scknife_final_celltype","interpretation":"descriptive cell-level distributions","inferential_tests":False},"patient_program_panel":{"panel":"F","grouping":"P1-P7","programs":R5_PROGRAM_IDS,"scale":"column z-score","cell_weighted":True,"inferential_tests":False},"association_flow_panel":supplementary_association_flow_metadata,"panels":supp_manifest,"program_dictionary_panel":"A","program_names_provisional":True}; (_compiled_dir/"Figure_5_supplementary_panel_manifest.json").write_text(json.dumps(supp_m,indent=2)+"\n")

# Register notebook-derived audit products in the Figure 5 bundle inventory.
import hashlib as R5F_hashlib
bundle_manifest_path = NOTEBOOK_DIR / "data/Figure_5/manifest.json"
bundle_manifest = json.loads(bundle_manifest_path.read_text())
bundle_files = []
for path in sorted(bundle_manifest_path.parent.rglob("*")):
    if path.is_file() and path != bundle_manifest_path:
        bundle_files.append({
            "name": path.relative_to(bundle_manifest_path.parent).as_posix(),
            "bytes": path.stat().st_size,
            "sha256": R5F_hashlib.sha256(path.read_bytes()).hexdigest(),
        })
bundle_manifest["files"] = bundle_files
bundle_manifest_path.write_text(json.dumps(bundle_manifest, indent=2) + "\n")
print("Figure 5 main, supplementary, standalone, and derived exports complete.")